# # 🎮 Reinforcement Learning Lab — 12 Experiments in One Notebook
#
# Each experiment lives in **one code cell** (preceded by a short explanation cell), and every code cell is split into
# commented sections: `SETUP → ENVIRONMENT → ALGORITHM → TRAINING → VISUALIZATION → RESULTS`.
#
# | # | Experiment | Family |
# |---|-----------|--------|
# | 1 | Epsilon-greedy ad selection | Bandits |
# | 2 | State-value function estimation (DP, TD(0), MC) | Prediction |
# | 3 | Policy iteration | Dynamic programming |
# | 4 | Monte Carlo prediction (Blackjack) | Model-free prediction |
# | 5 | Q-learning (Cliff Walking) | Off-policy TD control |
# | 6 | SARSA (Cliff Walking) | On-policy TD control |
# | 7 | Deep Q-Network on CartPole | Deep value-based |
# | 8 | TD-Gammon style TD(λ) self-play | Neural TD learning |
# | 9 | REINFORCE on CartPole | Policy gradient |
# | 10 | One-step Actor-Critic | Policy gradient + critic |
# | 11 | Advantage Actor-Critic (A2C, n-step, parallel envs) | Actor-critic methods |
# | 12 | Proximal Policy Optimization (PPO) | Trust-region-style policy gradient |
#
# **Install (once):** `pip install numpy matplotlib gymnasium torch`
#
# **Every experiment cell is fully independent**: it has its own imports, seeds, helper functions and environment, so you can run any one cell on its own, in any order.
(After each cell there is a paragraph describing what was done and how to read the results; each cell also prints its own results paragraph with this run's numbers.)

## Experiment 1 — Epsilon-Greedy for Choosing Advertisements (Multi-Armed Bandit)

**Problem.** A website has 10 candidate ads. Each ad has an *unknown* click-through rate (CTR). Every time we show an ad we get
reward 1 (click) or 0 (no click). We want to maximise total clicks.

**Exploration vs exploitation.** Always showing the ad that *looks* best (greedy) can lock us onto a bad ad early. Epsilon-greedy
shows a random ad with probability ε and the current best ad with probability 1-ε.

**Estimate.** Sample-average update: `Q(a) ← Q(a) + (R − Q(a)) / N(a)`.

**What to look for.** ε=0 gets stuck, ε too big wastes impressions, a small ε (or a *decaying* ε) does best in the long run.

In [ ]:
# ============================================================
# EXPERIMENT 1: EPSILON-GREEDY ADVERTISEMENT SELECTION
# ============================================================
# ---- STANDALONE SETUP (this cell needs nothing from any other cell) ----
import random, time, warnings                 # stdlib: randomness, timing, silence noisy warnings
import numpy as np                            # numerical arrays
import matplotlib.pyplot as plt               # all graphs
from collections import deque                 # replay buffers / fixed-length memories
warnings.filterwarnings("ignore")
SEED = 42                                     # fixed seed -> reproducible results
random.seed(SEED); np.random.seed(SEED)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})

# ---- SHARED HELPER: moving average to make noisy curves readable ----
def smooth(x, k=20):
    """Moving average used to make noisy reward curves readable."""
    x = np.asarray(x, dtype=float)
    if len(x) < k: return x
    return np.convolve(x, np.ones(k) / k, mode="valid")



# ---------- 1.1 SETTINGS ----------
N_ADS, STEPS, RUNS = 10, 3000, 100            # 10 ads, 3000 impressions per run, 100 independent runs
configs = {"ε = 0 (greedy)": 0.0, "ε = 0.01": 0.01, "ε = 0.1": 0.1, "ε = 0.3": 0.3, "ε decaying 1/√t": "decay"}


# ---------- 1.2 ALGORITHM ----------
def run_bandit(eps_cfg, seed=0):
    rng = np.random.default_rng(seed)
    reward_curve, optimal_curve = np.zeros(STEPS), np.zeros(STEPS)
    for _ in range(RUNS):
        true_ctr = rng.uniform(0.02, 0.30, N_ADS)           # hidden CTRs, new ads each run
        best_ad = true_ctr.argmax()
        Q, N = np.zeros(N_ADS), np.zeros(N_ADS)              # value estimates and pull counts
        for t in range(STEPS):
            eps = 1 / np.sqrt(t + 1) if eps_cfg == "decay" else eps_cfg
            if rng.random() < eps: a = rng.integers(N_ADS)                      # explore
            else: a = rng.choice(np.flatnonzero(Q == Q.max()))                  # exploit (random tie-break)
            r = float(rng.random() < true_ctr[a])                               # user clicks?
            N[a] += 1; Q[a] += (r - Q[a]) / N[a]                                # incremental mean
            reward_curve[t] += r; optimal_curve[t] += (a == best_ad)
    return reward_curve / RUNS, 100 * optimal_curve / RUNS


# ---------- 1.3 TRAINING / SIMULATION ----------
results = {name: run_bandit(cfg) for name, cfg in configs.items()}

# one detailed run to show how estimates converge to the true CTRs (ε = 0.1)
rng = np.random.default_rng(7)
true_ctr = rng.uniform(0.02, 0.30, N_ADS); Q, N = np.zeros(N_ADS), np.zeros(N_ADS)
for t in range(STEPS):
    a = rng.integers(N_ADS) if rng.random() < 0.1 else rng.choice(np.flatnonzero(Q == Q.max()))
    r = float(rng.random() < true_ctr[a]); N[a] += 1; Q[a] += (r - Q[a]) / N[a]

# ---------- 1.4 VISUALIZATION ----------
fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
for name, (rew, opt) in results.items():
    ax[0].plot(smooth(rew, 50), label=name); ax[1].plot(smooth(opt, 50), label=name)
ax[0].set(title="Average reward (click rate) over time", xlabel="Impressions", ylabel="Average reward"); ax[0].legend()
ax[1].set(title="% of impressions showing the best ad", xlabel="Impressions", ylabel="% optimal ad")
x = np.arange(N_ADS)
ax[2].bar(x - 0.2, true_ctr, 0.4, label="True CTR"); ax[2].bar(x + 0.2, Q, 0.4, label="Estimated Q")
ax[2].set(title="One run (ε=0.1): estimated vs true CTR", xlabel="Ad #", ylabel="CTR"); ax[2].legend()
plt.tight_layout(); plt.show()

# ---------- 1.5 RESULTS ----------
print(f"{'Strategy':<20}{'Avg reward (last 500)':>24}{'% optimal (last 500)':>24}")
for name, (rew, opt) in results.items():
    print(f"{name:<20}{rew[-500:].mean():>24.4f}{opt[-500:].mean():>24.1f}")

# ---------- 1.7 FINAL SUMMARY (printed paragraph with this run's numbers) ----------
import textwrap
final_r = {k: v[0][-500:].mean() for k, v in results.items()}
final_o = {k: v[1][-500:].mean() for k, v in results.items()}
best, worst = max(final_r, key=final_r.get), min(final_r, key=final_r.get)
print("\n" + "=" * 100 + "\nSUMMARY\n" + "=" * 100)
print(textwrap.fill(
    f"We simulated an ad server with {N_ADS} ads whose click-through rates were hidden from the agent, and compared five exploration "
    f"strategies over {STEPS} impressions, averaged over {RUNS} independent runs. After learning, the best strategy was '{best}' with an "
    f"average click rate of {final_r[best]:.3f} and {final_o[best]:.1f}% of impressions showing the truly best ad, while the weakest was "
    f"'{worst}' with {final_r[worst]:.3f} ({final_o[worst]:.1f}% optimal). Pure greedy (ε=0) reached only {final_o['ε = 0 (greedy)']:.1f}% optimal "
    f"because it locks onto whichever ad got an early lucky click; ε=0.3 keeps wasting impressions on random ads so its ceiling is lower. "
    f"The bar chart shows the sample-average estimates converging towards the real CTRs for ads that were shown often.", 100))

**Summary of Experiment 1.** WHAT WE DID: We built a bandit simulator with 10 ads whose click-through rates are unknown, then ran epsilon-
greedy agents with different ε values (including a decaying ε) for thousands of impressions, averaged over
many independent runs. The agent keeps a running average (sample-average update) of each ad's reward and
either explores a random ad with probability ε or exploits the best-looking ad. HOW TO READ THE PLOTS: the
left plot is average reward, the middle plot is the percentage of impressions that show the truly best ad, and
the right plot compares estimated and true CTR in a single run. RESULT: the printed summary gives this run's
numbers. In general ε=0 gets stuck on a mediocre ad, large ε wastes impressions, and a small or decaying ε
gives the best long-run click rate.

## Experiment 2 — State-Value Function Estimation

The **state-value function** `V^π(s)` is the expected return when starting in `s` and following policy π.
We estimate it three ways:

1. **Iterative policy evaluation (DP)** — uses the known model: `V(s) ← Σ_a π(a|s) [r + γ V(s')]`
2. **TD(0)** — learns from experience, bootstraps: `V(s) ← V(s) + α [r + γ V(s') − V(s)]`
3. **Every-visit Monte Carlo** — waits for the episode end: `V(s) ← V(s) + α [G − V(s)]`

**Part A:** 4×4 gridworld, random policy (exact answer from DP).
**Part B:** the classic 5-state *random walk* where the true values are 1/6 … 5/6, so we can measure TD vs MC error.

In [ ]:
# ============================================================
# EXPERIMENT 2: STATE VALUE FUNCTION ESTIMATION
# ============================================================
# ---- STANDALONE SETUP (this cell needs nothing from any other cell) ----
import random, time, warnings                 # stdlib: randomness, timing, silence noisy warnings
import numpy as np                            # numerical arrays
import matplotlib.pyplot as plt               # all graphs
from collections import deque                 # replay buffers / fixed-length memories
warnings.filterwarnings("ignore")
SEED = 42                                     # fixed seed -> reproducible results
random.seed(SEED); np.random.seed(SEED)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})

# ---- SHARED HELPER: moving average to make noisy curves readable ----
def smooth(x, k=20):
    """Moving average used to make noisy reward curves readable."""
    x = np.asarray(x, dtype=float)
    if len(x) < k: return x
    return np.convolve(x, np.ones(k) / k, mode="valid")

# ---- HELPER ENVIRONMENT: deterministic gridworld + heat-map drawing ----
# ---------- Gridworld (used by experiments 2 and 3) ----------
class GridWorld:
    """n x n deterministic grid. Two terminal corners (top-left, bottom-right).
    Every move costs -1, so the value of a state = -(expected steps to reach a terminal)."""
    ACTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]       # up, down, left, right
    ARROWS = ["↑", "↓", "←", "→"]

    def __init__(self, n=4):
        self.n, self.nS, self.nA = n, n * n, 4
        self.terminals = {0, self.nS - 1}

    def step(self, s, a):
        if s in self.terminals: return s, 0.0
        r, c = divmod(s, self.n)
        dr, dc = self.ACTIONS[a]
        r2, c2 = min(max(r + dr, 0), self.n - 1), min(max(c + dc, 0), self.n - 1)
        return r2 * self.n + c2, -1.0


def draw_values(ax, V, env, title, policy_text=None):
    """Heat-map of a state-value function with numbers (and optional arrows) in the cells."""
    grid = V.reshape(env.n, env.n)
    ax.imshow(grid, cmap="viridis")
    for s in range(env.nS):
        r, c = divmod(s, env.n)
        txt = "T" if s in env.terminals else f"{V[s]:.1f}"
        if policy_text is not None and s not in env.terminals: txt += "\n" + policy_text[s]
        ax.text(c, r, txt, ha="center", va="center", color="w", fontsize=9, fontweight="bold")
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)



# ---------- 2.1 PART A: DP POLICY EVALUATION ON GRIDWORLD ----------
env = GridWorld(4)
gamma = 1.0
V = np.zeros(env.nS)
deltas = []
while True:
    V_new = np.zeros(env.nS)
    for s in range(env.nS):
        if s in env.terminals: continue
        for a in range(env.nA):                          # equiprobable random policy
            s2, r = env.step(s, a)
            V_new[s] += 0.25 * (r + gamma * V[s2])
    delta = np.abs(V_new - V).max(); deltas.append(delta); V = V_new
    if delta < 1e-8: break
print(f"Policy evaluation converged in {len(deltas)} sweeps")

# ---------- 2.2 PART B: TD(0) vs MC ON RANDOM WALK ----------
# states 0..6 ; 0 and 6 are terminal ; reward +1 only when reaching state 6
true_V = np.arange(1, 6) / 6


def random_walk_episode():
    s, traj = 3, [3]
    while s not in (0, 6):
        s += random.choice([-1, 1]); traj.append(s)
    return traj, (1.0 if s == 6 else 0.0)


def run_td(alpha, episodes=100):
    V = np.full(7, 0.5); V[0] = V[6] = 0.0; errs = []
    for _ in range(episodes):
        traj, G = random_walk_episode()
        for i in range(len(traj) - 1):                    # TD(0) update after every step
            s, s2 = traj[i], traj[i + 1]
            r = G if s2 == 6 else 0.0
            V[s] += alpha * (r + V[s2] - V[s])
        errs.append(np.sqrt(np.mean((V[1:6] - true_V) ** 2)))
    return np.array(errs), V


def run_mc(alpha, episodes=100):
    V = np.full(7, 0.5); V[0] = V[6] = 0.0; errs = []
    for _ in range(episodes):
        traj, G = random_walk_episode()
        for s in traj[:-1]: V[s] += alpha * (G - V[s])    # MC update uses the full return
        errs.append(np.sqrt(np.mean((V[1:6] - true_V) ** 2)))
    return np.array(errs), V


# ---------- 2.3 TRAINING (averaged over 100 runs) ----------
td_alphas, mc_alphas, RUNS = [0.05, 0.10, 0.15], [0.01, 0.02, 0.04], 100
td_err = {a: np.mean([run_td(a)[0] for _ in range(RUNS)], axis=0) for a in td_alphas}
mc_err = {a: np.mean([run_mc(a)[0] for _ in range(RUNS)], axis=0) for a in mc_alphas}
V_td = run_td(0.05, 300)[1]; V_mc = run_mc(0.02, 300)[1]

# ---------- 2.4 VISUALIZATION ----------
fig, ax = plt.subplots(1, 4, figsize=(22, 4.5))
draw_values(ax[0], V, env, "A) True V of random policy (DP)")
ax[1].semilogy(deltas); ax[1].set(title="DP convergence (max change per sweep)", xlabel="Sweep", ylabel="Δ (log)")
for a, e in td_err.items(): ax[2].plot(e, label=f"TD(0) α={a}")
for a, e in mc_err.items(): ax[2].plot(e, "--", label=f"MC α={a}")
ax[2].set(title="B) Random walk: RMS error vs episodes", xlabel="Episodes", ylabel="RMS error"); ax[2].legend(fontsize=8)
ax[3].plot(range(1, 6), true_V, "k-o", label="True V"); ax[3].plot(range(1, 6), V_td[1:6], "-s", label="TD(0) after 300 ep")
ax[3].plot(range(1, 6), V_mc[1:6], "-^", label="MC after 300 ep"); ax[3].set_xticks(range(1, 6)); ax[3].set_xticklabels("ABCDE")
ax[3].set(title="Estimated vs true state values", xlabel="State", ylabel="V(s)"); ax[3].legend()
plt.tight_layout(); plt.show()

# ---------- 2.5 RESULTS ----------
print("DP values (rows of the 4x4 grid):\n", V.reshape(4, 4).round(1))
print("True random-walk values:", true_V.round(3)); print("TD(0) estimate :", V_td[1:6].round(3)); print("MC estimate    :", V_mc[1:6].round(3))

# ---------- 2.6 FINAL SUMMARY ----------
import textwrap
print("\n" + "=" * 100 + "\nSUMMARY\n" + "=" * 100)
print(textwrap.fill(
    f"Part A used dynamic programming (iterative policy evaluation) on a 4x4 gridworld with a random policy; it needed {len(deltas)} sweeps to converge "
    f"and gives the exact values (corner-adjacent states ≈ {V[1]:.0f}, farthest states ≈ {V[3]:.0f}) meaning 'expected steps to reach a terminal'. "
    f"Part B compared TD(0) and Monte Carlo on the 5-state random walk. After 100 episodes the average RMS error was {td_err[0.05][-1]:.3f} for TD(0) (α=0.05) "
    f"versus {mc_err[0.02][-1]:.3f} for MC (α=0.02). TD(0) usually learns faster early on because it bootstraps from neighbouring estimates instead of waiting "
    f"for the final outcome; MC is unbiased but noisier. After 300 episodes TD(0) estimated {np.round(V_td[1:6], 2)} and MC {np.round(V_mc[1:6], 2)} against the true {np.round(true_V, 2)}.", 100))

**Summary of Experiment 2.** WHAT WE DID: We estimated the state-value function V(s) in three ways. Dynamic programming sweeps the Bellman
expectation equation over a known model and gives the exact answer on the 4x4 grid. TD(0) and Monte Carlo
learn from sampled episodes on a 5-state random walk where the true values 1/6..5/6 are known, so we can
measure their error. HOW TO READ THE PLOTS: the heat-map shows how far each cell is from a terminal (more
negative means farther); the middle plot shows DP convergence and the RMS-error curves; the last plot overlays
estimated and true values. RESULT: see the printed summary. TD(0) typically drops its error faster than Monte
Carlo early on; both approach the true values with more episodes.

## Experiment 3 — Policy Iteration

Policy iteration alternates two steps until the policy stops changing:

1. **Policy evaluation:** compute `V^π` for the current policy (iterate the Bellman expectation equation).
2. **Policy improvement:** make the policy greedy: `π'(s) = argmax_a [r + γ V^π(s')]`.

Each improvement is guaranteed to be at least as good, and in finite MDPs it converges to the **optimal policy** in a handful of iterations.
We run it on a 5×5 gridworld starting from a *random* deterministic policy and plot the policy at several stages.

In [ ]:
# ============================================================
# EXPERIMENT 3: POLICY ITERATION
# ============================================================
# ---- STANDALONE SETUP (this cell needs nothing from any other cell) ----
import random, time, warnings                 # stdlib: randomness, timing, silence noisy warnings
import numpy as np                            # numerical arrays
import matplotlib.pyplot as plt               # all graphs
from collections import deque                 # replay buffers / fixed-length memories
warnings.filterwarnings("ignore")
SEED = 42                                     # fixed seed -> reproducible results
random.seed(SEED); np.random.seed(SEED)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})

# ---- SHARED HELPER: moving average to make noisy curves readable ----
def smooth(x, k=20):
    """Moving average used to make noisy reward curves readable."""
    x = np.asarray(x, dtype=float)
    if len(x) < k: return x
    return np.convolve(x, np.ones(k) / k, mode="valid")

# ---- HELPER ENVIRONMENT: deterministic gridworld + heat-map drawing ----
# ---------- Gridworld (used by experiments 2 and 3) ----------
class GridWorld:
    """n x n deterministic grid. Two terminal corners (top-left, bottom-right).
    Every move costs -1, so the value of a state = -(expected steps to reach a terminal)."""
    ACTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]       # up, down, left, right
    ARROWS = ["↑", "↓", "←", "→"]

    def __init__(self, n=4):
        self.n, self.nS, self.nA = n, n * n, 4
        self.terminals = {0, self.nS - 1}

    def step(self, s, a):
        if s in self.terminals: return s, 0.0
        r, c = divmod(s, self.n)
        dr, dc = self.ACTIONS[a]
        r2, c2 = min(max(r + dr, 0), self.n - 1), min(max(c + dc, 0), self.n - 1)
        return r2 * self.n + c2, -1.0


def draw_values(ax, V, env, title, policy_text=None):
    """Heat-map of a state-value function with numbers (and optional arrows) in the cells."""
    grid = V.reshape(env.n, env.n)
    ax.imshow(grid, cmap="viridis")
    for s in range(env.nS):
        r, c = divmod(s, env.n)
        txt = "T" if s in env.terminals else f"{V[s]:.1f}"
        if policy_text is not None and s not in env.terminals: txt += "\n" + policy_text[s]
        ax.text(c, r, txt, ha="center", va="center", color="w", fontsize=9, fontweight="bold")
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)



# ---------- 3.1 ENVIRONMENT ----------
env = GridWorld(5)
gamma = 0.95
rng = np.random.default_rng(3)
policy = rng.integers(0, 4, env.nS)               # random initial deterministic policy


# ---------- 3.2 ALGORITHM ----------
def policy_evaluation(policy, V, theta=1e-8):
    """Iterate the Bellman expectation backup for a fixed policy until convergence."""
    sweeps = 0
    while True:
        delta = 0
        for s in range(env.nS):
            if s in env.terminals: continue
            s2, r = env.step(s, policy[s])
            v_new = r + gamma * V[s2]
            delta = max(delta, abs(v_new - V[s])); V[s] = v_new      # in-place (Gauss-Seidel) sweep
        sweeps += 1
        if delta < theta: return V, sweeps


def policy_improvement(V):
    """Greedy policy w.r.t. V. Returns new policy and arrows string (shows all tied best actions)."""
    new_pol, arrows = np.zeros(env.nS, dtype=int), [""] * env.nS
    for s in range(env.nS):
        if s in env.terminals: continue
        q = np.array([env.step(s, a)[1] + gamma * V[env.step(s, a)[0]] for a in range(4)])
        new_pol[s] = int(np.argmax(q))
        arrows[s] = "".join(env.ARROWS[a] for a in range(4) if q[a] > q.max() - 1e-9)
    return new_pol, arrows


# ---------- 3.3 TRAINING LOOP ----------
V = np.zeros(env.nS)
snapshots = [(0, V.copy(), [env.ARROWS[a] for a in policy])]    # (iteration, V, arrows) for plotting
history = []
for it in range(1, 50):
    V, sweeps = policy_evaluation(policy, V)                     # step 1: evaluate
    new_policy, arrows = policy_improvement(V)                   # step 2: improve
    changed = int((new_policy != policy)[[s for s in range(env.nS) if s not in env.terminals]].sum())
    history.append((it, sweeps, changed)); snapshots.append((it, V.copy(), arrows))
    print(f"Iteration {it}: evaluation sweeps={sweeps:3d}, states whose action changed={changed}")
    if changed == 0: print("✅ Policy is stable -> optimal policy found"); break
    policy = new_policy

# ---------- 3.4 VISUALIZATION ----------
show = [snapshots[0], snapshots[1], snapshots[len(snapshots) // 2], snapshots[-1]]
fig, ax = plt.subplots(1, 5, figsize=(26, 5))
for a, (it, Vs, arr) in zip(ax[:4], show):
    draw_values(a, Vs, env, "Initial random policy (V=0)" if it == 0 else f"After iteration {it}", arr)
ax[4].bar([h[0] for h in history], [h[2] for h in history], color="tab:orange")
ax[4].set(title="Policy changes per iteration", xlabel="Iteration", ylabel="# states changed")
plt.tight_layout(); plt.show()

# ---------- 3.5 RESULTS ----------
print("Optimal V(s):\n", V.reshape(5, 5).round(2))
print("Optimal policy:"); print(np.array(snapshots[-1][2]).reshape(5, 5))

# ---------- 3.6 FINAL SUMMARY ----------
import textwrap
print("\n" + "=" * 100 + "\nSUMMARY\n" + "=" * 100)
print(textwrap.fill(
    f"Policy iteration started from a random deterministic policy on a 5x5 gridworld (γ={gamma}) and alternated full policy evaluation with greedy improvement. "
    f"It converged after {len(history)} iterations: the number of states whose action changed per iteration was {[h[2] for h in history]}, reaching zero when the policy "
    f"became stable. The first evaluation needed {history[0][1]} sweeps because the random policy wanders, but later evaluations needed only {history[-1][1]} because V was already close. "
    f"The final value function decreases smoothly with distance from the two terminal corners (farthest state V = {V.min():.2f}), and the arrows point along shortest paths to the nearest terminal, "
    f"which is the optimal policy for this MDP.", 100))

**Summary of Experiment 3.** WHAT WE DID: We ran policy iteration on a 5x5 gridworld: start with a random policy, evaluate it until V
converges, make the policy greedy with respect to V, and repeat until no action changes. HOW TO READ THE
PLOTS: the four heat-maps show the value function and the arrows of the policy at the start, after the first
iteration, in the middle and at the end; the bar chart shows how many states changed their action in each
iteration. RESULT: the policy stabilises in only a few iterations and the final arrows point along shortest
paths toward the nearest terminal corner; ties are shown with several arrows because several actions are
equally optimal.

## Experiment 4 — Monte Carlo Prediction (Blackjack)

**Monte Carlo prediction** estimates `V^π(s)` by *averaging the returns* observed after visiting `s` over many complete episodes.
No model of the environment is needed — only sampled episodes.

**Game.** Simplified casino Blackjack (infinite deck). State = (player sum 12-21, dealer's showing card, usable ace?).
**Policy evaluated:** stick only on 20 or 21, otherwise hit. We use **first-visit MC** and compare 10,000 vs 500,000 episodes —
more episodes → smoother, more accurate surface.

In [ ]:
# ============================================================
# EXPERIMENT 4: MONTE CARLO PREDICTION (BLACKJACK, FIRST-VISIT)
# ============================================================
# ---- STANDALONE SETUP (this cell needs nothing from any other cell) ----
import random, time, warnings                 # stdlib: randomness, timing, silence noisy warnings
import numpy as np                            # numerical arrays
import matplotlib.pyplot as plt               # all graphs
from collections import deque                 # replay buffers / fixed-length memories
warnings.filterwarnings("ignore")
SEED = 42                                     # fixed seed -> reproducible results
random.seed(SEED); np.random.seed(SEED)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})

# ---- SHARED HELPER: moving average to make noisy curves readable ----
def smooth(x, k=20):
    """Moving average used to make noisy reward curves readable."""
    x = np.asarray(x, dtype=float)
    if len(x) < k: return x
    return np.convolve(x, np.ones(k) / k, mode="valid")


from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers the 3D projection)


# ---------- 4.1 ENVIRONMENT (self-contained blackjack) ----------
def draw_card(): return min(random.randint(1, 13), 10)           # 1 = ace, face cards count 10


def hand_value(cards):
    total = sum(cards)
    if 1 in cards and total + 10 <= 21: return total + 10, True  # an ace counted as 11 = "usable"
    return total, False


def play_episode():
    """Play one hand with the fixed policy. Returns visited states and the final reward."""
    player, dealer = [draw_card(), draw_card()], [draw_card(), draw_card()]
    states = []
    while True:
        total, usable = hand_value(player)
        if total >= 12: states.append((total, dealer[0], usable))   # only record decision states
        if total >= 20 or total > 21: break                            # policy: stick on 20/21
        player.append(draw_card())
        if hand_value(player)[0] > 21: return states, -1.0             # player busts
    while hand_value(dealer)[0] < 17: dealer.append(draw_card())       # dealer hits below 17
    p, d = hand_value(player)[0], hand_value(dealer)[0]
    return states, (1.0 if d > 21 or p > d else (0.0 if p == d else -1.0))


# ---------- 4.2 ALGORITHM: FIRST-VISIT MC ----------
def mc_prediction(n_episodes):
    returns_sum, returns_cnt = {}, {}
    for _ in range(n_episodes):
        states, G = play_episode()               # reward only at the end, gamma=1 => G is the same for all states
        for s in dict.fromkeys(states):          # dict.fromkeys keeps only the FIRST visit of each state
            returns_sum[s] = returns_sum.get(s, 0.0) + G
            returns_cnt[s] = returns_cnt.get(s, 0) + 1
    return {s: returns_sum[s] / returns_cnt[s] for s in returns_sum}


# ---------- 4.3 TRAINING ----------
t0 = time.time()
V_10k = mc_prediction(10_000)
V_500k = mc_prediction(500_000)
print(f"MC prediction done in {time.time() - t0:.1f}s")


# ---------- 4.4 VISUALIZATION ----------
def to_grid(V, usable):
    sums, dealer = np.arange(12, 22), np.arange(1, 11)
    Z = np.array([[V.get((p, d, usable), np.nan) for d in dealer] for p in sums])
    return np.meshgrid(dealer, sums), Z


fig = plt.figure(figsize=(16, 11))
for i, (V_, name) in enumerate([(V_10k, "10,000 episodes"), (V_500k, "500,000 episodes")]):
    for j, usable in enumerate([True, False]):
        ax = fig.add_subplot(2, 2, 2 * i + j + 1, projection="3d")
        (X, Y), Z = to_grid(V_, usable)
        ax.plot_surface(X, Y, Z, cmap="coolwarm", vmin=-1, vmax=1)
        ax.set(title=f"{name} — {'usable ace' if usable else 'no usable ace'}", xlabel="Dealer showing", ylabel="Player sum", zlabel="V(s)")
        ax.set_zlim(-1, 1); ax.view_init(30, -120)
plt.tight_layout(); plt.show()

# ---------- 4.5 RESULTS ----------
print("V(player=20, dealer=10, no ace) =", round(V_500k[(20, 10, False)], 3), "(sticking on 20 is strong)")
print("V(player=12, dealer=10, no ace) =", round(V_500k[(12, 10, False)], 3), "(hitting on 12 vs dealer 10 is weak)")
print("Number of states estimated:", len(V_500k))

# ---------- 4.6 FINAL SUMMARY ----------
import textwrap
print("\n" + "=" * 100 + "\nSUMMARY\n" + "=" * 100)
print(textwrap.fill(
    f"We estimated V(s) for the 'stick on 20 or 21' Blackjack policy using first-visit Monte Carlo, averaging returns over 10,000 and then 500,000 simulated hands, with no model of the game. "
    f"{len(V_500k)} states were visited. With 500,000 episodes V(20, dealer 10, no ace) = {V_500k[(20, 10, False)]:.3f} (strong, we usually win) while V(12, dealer 10, no ace) = {V_500k[(12, 10, False)]:.3f} "
    f"(weak, because this policy keeps hitting and often busts). The 10,000-episode surfaces are visibly rough, especially for usable-ace states which are rarer, while the 500,000-episode surfaces are smooth: "
    f"the law of large numbers makes the average return converge to the true state value.", 100))

**Summary of Experiment 4.** WHAT WE DID: We wrote a self-contained Blackjack simulator and used first-visit Monte Carlo prediction to
estimate V(s) for the fixed policy 'stick only on 20 or 21'. The state is (player sum, dealer's showing card,
usable ace). For each hand we record the first visit of each state and average the final reward over all
episodes. HOW TO READ THE PLOTS: 3D surfaces of V against dealer card and player sum, with and without a
usable ace, after 10,000 and 500,000 episodes. RESULT: the larger run is much smoother and shows high values
for player sums of 20 and 21 and low values for 12 to 16 (this policy hits on those and often busts). The
printed summary gives exact numbers.

## Experiment 5 — Q-Learning (Off-Policy TD Control) on Cliff Walking

**Update:** `Q(s,a) ← Q(s,a) + α [ r + γ · max_a' Q(s',a') − Q(s,a) ]`

It is **off-policy**: the behaviour policy is ε-greedy (to explore) but the target uses the *greedy* `max`, so Q-learning learns the
value of the **optimal** policy. In Cliff Walking the optimal path hugs the cliff edge (shortest), but with ε-greedy exploration the agent
sometimes falls — so its *online* reward is lower than a safer path would give.

In [ ]:
# ============================================================
# EXPERIMENT 5: Q-LEARNING ON CLIFF WALKING
# ============================================================
# ---- STANDALONE SETUP (this cell needs nothing from any other cell) ----
import random, time, warnings                 # stdlib: randomness, timing, silence noisy warnings
import numpy as np                            # numerical arrays
import matplotlib.pyplot as plt               # all graphs
from collections import deque                 # replay buffers / fixed-length memories
warnings.filterwarnings("ignore")
SEED = 42                                     # fixed seed -> reproducible results
random.seed(SEED); np.random.seed(SEED)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})

# ---- SHARED HELPER: moving average to make noisy curves readable ----
def smooth(x, k=20):
    """Moving average used to make noisy reward curves readable."""
    x = np.asarray(x, dtype=float)
    if len(x) < k: return x
    return np.convolve(x, np.ones(k) / k, mode="valid")

# ---- HELPER ENVIRONMENT: Cliff Walking + policy drawing ----
# ---------- Cliff Walking (used by experiments 5 and 6) ----------
class CliffWalking:
    """4x12 grid. Start bottom-left, goal bottom-right, cells between them are a cliff.
    Reward -1 per step, -100 (and teleport to start) when stepping on the cliff."""
    ACTIONS = [(-1, 0), (0, 1), (1, 0), (0, -1)]       # up, right, down, left
    ARROWS = ["↑", "→", "↓", "←"]

    def __init__(self):
        self.H, self.W = 4, 12
        self.start, self.goal = (3, 0), (3, 11)
        self.nS, self.nA = self.H * self.W, 4

    def reset(self):
        self.pos = self.start
        return self.pos[0] * self.W + self.pos[1]

    def step(self, a):
        r, c = self.pos
        dr, dc = self.ACTIONS[a]
        r, c = min(max(r + dr, 0), self.H - 1), min(max(c + dc, 0), self.W - 1)
        reward, done = -1.0, False
        if r == 3 and 1 <= c <= 10:                     # fell off the cliff
            reward, (r, c) = -100.0, self.start
        if (r, c) == self.goal: done = True
        self.pos = (r, c)
        return r * self.W + c, reward, done


def draw_cliff_policy(ax, Q, env, title):
    """Draw the greedy policy as arrows and trace the greedy path from start to goal."""
    ax.set_xlim(0, env.W); ax.set_ylim(env.H, 0); ax.set_aspect("equal"); ax.grid(False)
    for c in range(1, 11): ax.add_patch(plt.Rectangle((c, 3), 1, 1, color="#444"))
    ax.add_patch(plt.Rectangle((0, 3), 1, 1, color="#8fd18f")); ax.add_patch(plt.Rectangle((11, 3), 1, 1, color="#f2c14e"))
    for r in range(env.H):
        for c in range(env.W):
            if r == 3 and c >= 1: continue
            ax.text(c + .5, r + .5, env.ARROWS[int(np.argmax(Q[r * env.W + c]))], ha="center", va="center", fontsize=13)
    # greedy path
    pos, path = env.start, [env.start]
    for _ in range(60):
        a = int(np.argmax(Q[pos[0] * env.W + pos[1]])); dr, dc = env.ACTIONS[a]
        pos = (min(max(pos[0] + dr, 0), 3), min(max(pos[1] + dc, 0), 11)); path.append(pos)
        if pos == env.goal or (pos[0] == 3 and 1 <= pos[1] <= 10): break
    ax.plot([p[1] + .5 for p in path], [p[0] + .5 for p in path], "r-o", lw=2, ms=4)
    ax.set_title(title); ax.set_xticks(range(env.W + 1)); ax.set_yticks(range(env.H + 1))



# ---------- 5.1 HYPERPARAMETERS ----------
EPISODES, ALPHA, GAMMA, EPS = 500, 0.5, 1.0, 0.1


# ---------- 5.2 ALGORITHM ----------
def epsilon_greedy(Q, s, eps, rng):
    if rng.random() < eps: return int(rng.integers(Q.shape[1]))
    return int(rng.choice(np.flatnonzero(Q[s] == Q[s].max())))


def train_qlearning(episodes=EPISODES, seed=0):
    rng, env = np.random.default_rng(seed), CliffWalking()
    Q, ep_rewards = np.zeros((env.nS, env.nA)), []
    for _ in range(episodes):
        s, total, done, steps = env.reset(), 0.0, False, 0
        while not done and steps < 500:
            a = epsilon_greedy(Q, s, EPS, rng)                       # behaviour policy: ε-greedy
            s2, r, done = env.step(a)
            target = r + GAMMA * Q[s2].max() * (not done)            # target policy: GREEDY (max)
            Q[s, a] += ALPHA * (target - Q[s, a])                    # TD update
            s, total, steps = s2, total + r, steps + 1
        ep_rewards.append(total)
    return Q, np.array(ep_rewards)


# ---------- 5.3 TRAINING ----------
Q_q, _ = train_qlearning(seed=0)                                             # one run for the policy picture
all_runs_q = np.array([train_qlearning(seed=s)[1] for s in range(1, 21)])    # 20 runs for a smooth curve

# ---------- 5.4 VISUALIZATION ----------
env = CliffWalking()
fig, ax = plt.subplots(1, 3, figsize=(20, 4.5), gridspec_kw={"width_ratios": [1.1, 1.6, 1]})
ax[0].plot(all_runs_q.mean(0), alpha=.4, label="mean of 20 runs"); ax[0].plot(smooth(all_runs_q.mean(0), 10), label="smoothed")
ax[0].set(title="Q-learning: reward per episode", xlabel="Episode", ylabel="Sum of rewards", ylim=(-150, 0)); ax[0].legend()
draw_cliff_policy(ax[1], Q_q, env, "Greedy policy & path (red) — hugs the cliff edge")
im = ax[2].imshow(Q_q.max(1).reshape(4, 12), cmap="viridis"); ax[2].set_title("max_a Q(s,a)"); ax[2].grid(False); plt.colorbar(im, ax=ax[2], fraction=.04)
plt.tight_layout(); plt.show()

# ---------- 5.5 RESULTS ----------
print(f"Mean online reward, last 100 episodes (with exploration): {all_runs_q[:, -100:].mean():.2f}")
print("Optimal greedy path length is 13 steps (reward -13) — Q-learning learns it, but exploration near the cliff costs reward online.")

# ---------- 5.6 FINAL SUMMARY ----------
import textwrap
print("\n" + "=" * 100 + "\nSUMMARY\n" + "=" * 100)
print(textwrap.fill(
    f"Tabular Q-learning (α={ALPHA}, γ={GAMMA}, ε={EPS}) was trained for {EPISODES} episodes on a 4x12 Cliff Walking grid, averaged over 20 runs. "
    f"The average online reward over the last 100 episodes was {all_runs_q[:, -100:].mean():.1f}. The learned greedy policy follows the edge of the cliff, which is the shortest (13-step) route, "
    f"because the update bootstraps from max Q(s', a') — the value of acting greedily — and ignores that the ε-greedy behaviour policy may step off the cliff. That is why online reward is "
    f"worse than the greedy path would suggest: exploration near the cliff sometimes costs -100.", 100))

**Summary of Experiment 5.** WHAT WE DID: We implemented tabular Q-learning, an off-policy TD control method, on the Cliff Walking grid.
The agent behaves ε-greedily but updates Q(s,a) toward r + γ·max Q(s',a'), so it learns the value of the
greedy policy. HOW TO READ THE PLOTS: the left plot is the reward per episode averaged over 20 runs, the
middle plot draws the greedy arrows and the path from start to goal, and the right plot shows the maximum
Q-value in each cell. RESULT: the path hugs the cliff edge (shortest path) while the online reward during
training is lower than the optimal -13 because exploration occasionally drops the agent off the cliff. Numbers
are in the printed summary.

## Experiment 6 — SARSA (On-Policy TD Control) on Cliff Walking

**Update:** `Q(s,a) ← Q(s,a) + α [ r + γ · Q(s',a') − Q(s,a) ]` where `a'` is the action *actually chosen* by the ε-greedy policy
(name = **S**tate, **A**ction, **R**eward, next **S**tate, next **A**ction).

Because SARSA's target includes the exploratory action, it "knows" it may slip near the cliff, so it learns the **safer, longer path**
away from the edge. Result: better *online* reward than Q-learning, but a less aggressive final path.

*(Requires `train_qlearning` from Experiment 5 for the comparison.)*

In [ ]:
# ============================================================
# EXPERIMENT 6: SARSA ON CLIFF WALKING (+ comparison with Q-learning)
# ============================================================
# ---- STANDALONE SETUP (this cell needs nothing from any other cell) ----
import random, time, warnings                 # stdlib: randomness, timing, silence noisy warnings
import numpy as np                            # numerical arrays
import matplotlib.pyplot as plt               # all graphs
from collections import deque                 # replay buffers / fixed-length memories
warnings.filterwarnings("ignore")
SEED = 42                                     # fixed seed -> reproducible results
random.seed(SEED); np.random.seed(SEED)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})

# ---- SHARED HELPER: moving average to make noisy curves readable ----
def smooth(x, k=20):
    """Moving average used to make noisy reward curves readable."""
    x = np.asarray(x, dtype=float)
    if len(x) < k: return x
    return np.convolve(x, np.ones(k) / k, mode="valid")

# ---- HELPER ENVIRONMENT: Cliff Walking + policy drawing ----
# ---------- Cliff Walking (used by experiments 5 and 6) ----------
class CliffWalking:
    """4x12 grid. Start bottom-left, goal bottom-right, cells between them are a cliff.
    Reward -1 per step, -100 (and teleport to start) when stepping on the cliff."""
    ACTIONS = [(-1, 0), (0, 1), (1, 0), (0, -1)]       # up, right, down, left
    ARROWS = ["↑", "→", "↓", "←"]

    def __init__(self):
        self.H, self.W = 4, 12
        self.start, self.goal = (3, 0), (3, 11)
        self.nS, self.nA = self.H * self.W, 4

    def reset(self):
        self.pos = self.start
        return self.pos[0] * self.W + self.pos[1]

    def step(self, a):
        r, c = self.pos
        dr, dc = self.ACTIONS[a]
        r, c = min(max(r + dr, 0), self.H - 1), min(max(c + dc, 0), self.W - 1)
        reward, done = -1.0, False
        if r == 3 and 1 <= c <= 10:                     # fell off the cliff
            reward, (r, c) = -100.0, self.start
        if (r, c) == self.goal: done = True
        self.pos = (r, c)
        return r * self.W + c, reward, done


def draw_cliff_policy(ax, Q, env, title):
    """Draw the greedy policy as arrows and trace the greedy path from start to goal."""
    ax.set_xlim(0, env.W); ax.set_ylim(env.H, 0); ax.set_aspect("equal"); ax.grid(False)
    for c in range(1, 11): ax.add_patch(plt.Rectangle((c, 3), 1, 1, color="#444"))
    ax.add_patch(plt.Rectangle((0, 3), 1, 1, color="#8fd18f")); ax.add_patch(plt.Rectangle((11, 3), 1, 1, color="#f2c14e"))
    for r in range(env.H):
        for c in range(env.W):
            if r == 3 and c >= 1: continue
            ax.text(c + .5, r + .5, env.ARROWS[int(np.argmax(Q[r * env.W + c]))], ha="center", va="center", fontsize=13)
    # greedy path
    pos, path = env.start, [env.start]
    for _ in range(60):
        a = int(np.argmax(Q[pos[0] * env.W + pos[1]])); dr, dc = env.ACTIONS[a]
        pos = (min(max(pos[0] + dr, 0), 3), min(max(pos[1] + dc, 0), 11)); path.append(pos)
        if pos == env.goal or (pos[0] == 3 and 1 <= pos[1] <= 10): break
    ax.plot([p[1] + .5 for p in path], [p[0] + .5 for p in path], "r-o", lw=2, ms=4)
    ax.set_title(title); ax.set_xticks(range(env.W + 1)); ax.set_yticks(range(env.H + 1))


# ---------- 6.0 Q-LEARNING BASELINE (copied from experiment 5 so this cell is independent) — 5.1 HYPERPARAMETERS ----------
EPISODES, ALPHA, GAMMA, EPS = 500, 0.5, 1.0, 0.1


# ---------- 5.2 ALGORITHM ----------
def epsilon_greedy(Q, s, eps, rng):
    if rng.random() < eps: return int(rng.integers(Q.shape[1]))
    return int(rng.choice(np.flatnonzero(Q[s] == Q[s].max())))


def train_qlearning(episodes=EPISODES, seed=0):
    rng, env = np.random.default_rng(seed), CliffWalking()
    Q, ep_rewards = np.zeros((env.nS, env.nA)), []
    for _ in range(episodes):
        s, total, done, steps = env.reset(), 0.0, False, 0
        while not done and steps < 500:
            a = epsilon_greedy(Q, s, EPS, rng)                       # behaviour policy: ε-greedy
            s2, r, done = env.step(a)
            target = r + GAMMA * Q[s2].max() * (not done)            # target policy: GREEDY (max)
            Q[s, a] += ALPHA * (target - Q[s, a])                    # TD update
            s, total, steps = s2, total + r, steps + 1
        ep_rewards.append(total)
    return Q, np.array(ep_rewards)


# ---------- 5.3 TRAINING ----------
Q_q, _ = train_qlearning(seed=0)                                             # one run for the policy picture
all_runs_q = np.array([train_qlearning(seed=s)[1] for s in range(1, 21)])    # 20 runs for a smooth curve



# ---------- 6.1 ALGORITHM ----------
def train_sarsa(episodes=EPISODES, seed=0):
    rng, env = np.random.default_rng(seed), CliffWalking()
    Q, ep_rewards = np.zeros((env.nS, env.nA)), []
    for _ in range(episodes):
        s, done, total, steps = env.reset(), False, 0.0, 0
        a = epsilon_greedy(Q, s, EPS, rng)                           # choose first action BEFORE the loop
        while not done and steps < 500:
            s2, r, done = env.step(a)
            a2 = epsilon_greedy(Q, s2, EPS, rng)                     # next action from the SAME ε-greedy policy
            target = r + GAMMA * Q[s2, a2] * (not done)              # on-policy target uses Q(s',a')
            Q[s, a] += ALPHA * (target - Q[s, a])
            s, a, total, steps = s2, a2, total + r, steps + 1
        ep_rewards.append(total)
    return Q, np.array(ep_rewards)


# ---------- 6.2 TRAINING ----------
Q_s, _ = train_sarsa(seed=0)
all_runs_s = np.array([train_sarsa(seed=s)[1] for s in range(1, 21)])

# ---------- 6.3 VISUALIZATION ----------
env = CliffWalking()
fig, ax = plt.subplots(1, 3, figsize=(21, 4.5), gridspec_kw={"width_ratios": [1.1, 1.6, 1.6]})
ax[0].plot(smooth(all_runs_q.mean(0), 10), label="Q-learning"); ax[0].plot(smooth(all_runs_s.mean(0), 10), label="SARSA")
ax[0].set(title="Online reward per episode (mean of 20 runs)", xlabel="Episode", ylabel="Sum of rewards", ylim=(-100, 0)); ax[0].legend()
draw_cliff_policy(ax[1], Q_q, env, "Q-learning path: risky, shortest")
draw_cliff_policy(ax[2], Q_s, env, "SARSA path: safer, longer")
plt.tight_layout(); plt.show()

# ---------- 6.4 RESULTS ----------
print(f"Last-100-episode online reward  ->  Q-learning: {all_runs_q[:, -100:].mean():.2f} | SARSA: {all_runs_s[:, -100:].mean():.2f}")
print("SARSA wins online because it accounts for exploration; Q-learning wins on the *greedy* path length (13 steps).")

# ---------- 6.5 FINAL SUMMARY ----------
import textwrap
print("\n" + "=" * 100 + "\nSUMMARY\n" + "=" * 100)
print(textwrap.fill(
    f"SARSA was trained on the same Cliff Walking problem and compared to Q-learning (20 runs each, {EPISODES} episodes). Over the last 100 episodes the average online reward was "
    f"{all_runs_s[:, -100:].mean():.1f} for SARSA versus {all_runs_q[:, -100:].mean():.1f} for Q-learning. SARSA's target uses the action it will actually take next, so it learns that states next to the cliff are "
    f"dangerous under ε-greedy exploration and chooses a longer, safer path away from the edge; Q-learning learns the risky optimal path and pays for exploration. "
    f"If ε were decayed towards 0, both would converge to the same optimal path.", 100))

**Summary of Experiment 6.** WHAT WE DID: We implemented SARSA, an on-policy TD control method, on the same Cliff Walking grid, and re-ran
Q-learning inside this cell so that the cell works on its own. SARSA updates toward r + γ·Q(s',a') where a' is
the action actually chosen by the ε-greedy policy. HOW TO READ THE PLOTS: left is the online reward of both
algorithms, middle is the Q-learning path, and right is the SARSA path. RESULT: SARSA learns a safer, longer
route away from the cliff and therefore earns a higher online reward during training, while Q-learning finds
the shortest but riskier route. This is the classic on-policy versus off-policy difference; see the printed
summary for the measured numbers.

## Experiment 7 — Deep Q-Learning (DQN) on CartPole

**Task.** Balance a pole on a cart. State = 4 numbers (position, velocity, angle, angular velocity); actions = push left / right;
reward +1 per step survived (max 500 in `CartPole-v1`).

**DQN = Q-learning + a neural network + two stabilising tricks:**
1. **Experience replay** — store transitions, train on random mini-batches (breaks correlation between consecutive samples).
2. **Target network** — a slowly-updated copy of the Q-network is used to compute targets: `y = r + γ max_a' Q_target(s',a')`.

Loss: Huber loss between `Q(s,a)` and `y`. Exploration: ε decays from 1.0 → 0.05.

In [ ]:
# ============================================================
# EXPERIMENT 7: DEEP Q-NETWORK (DQN) FOR CARTPOLE
# ============================================================
# ---- STANDALONE SETUP (this cell needs nothing from any other cell) ----
import random, time, warnings                 # stdlib: randomness, timing, silence noisy warnings
import numpy as np                            # numerical arrays
import matplotlib.pyplot as plt               # all graphs
from collections import deque                 # replay buffers / fixed-length memories
warnings.filterwarnings("ignore")
SEED = 42                                     # fixed seed -> reproducible results
random.seed(SEED); np.random.seed(SEED)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})
import gymnasium as gym                       # CartPole environment
import torch                                  # neural networks + autograd
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical   # sampling actions from a policy
torch.set_num_threads(1)                      # tiny networks run fastest on 1 thread
torch.manual_seed(SEED)

# ---- SHARED HELPER: moving average to make noisy curves readable ----
def smooth(x, k=20):
    """Moving average used to make noisy reward curves readable."""
    x = np.asarray(x, dtype=float)
    if len(x) < k: return x
    return np.convolve(x, np.ones(k) / k, mode="valid")



# ---------- 7.1 HYPERPARAMETERS ----------
GAMMA, LR, BATCH, BUFFER = 0.99, 1e-3, 64, 50_000
EPS_START, EPS_END, EPS_DECAY_STEPS = 1.0, 0.05, 4_000
TARGET_SYNC, EPISODES, WARMUP = 250, 300, 500
torch.manual_seed(SEED)


# ---------- 7.2 MODEL + REPLAY BUFFER ----------
class QNet(nn.Module):
    def __init__(self, n_obs, n_act):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_obs, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, n_act))
    def forward(self, x): return self.net(x)


class ReplayBuffer:
    def __init__(self, cap): self.buf = deque(maxlen=cap)
    def push(self, *transition): self.buf.append(transition)
    def sample(self, n):
        s, a, r, s2, d = zip(*random.sample(self.buf, n))
        return (torch.tensor(np.array(s), dtype=torch.float32), torch.tensor(a).unsqueeze(1),
                torch.tensor(r, dtype=torch.float32), torch.tensor(np.array(s2), dtype=torch.float32), torch.tensor(d, dtype=torch.float32))
    def __len__(self): return len(self.buf)


env = gym.make("CartPole-v1")
n_obs, n_act = env.observation_space.shape[0], env.action_space.n
q_net, target_net = QNet(n_obs, n_act), QNet(n_obs, n_act)
target_net.load_state_dict(q_net.state_dict())
optimizer, buffer = optim.Adam(q_net.parameters(), lr=LR), ReplayBuffer(BUFFER)

# ---------- 7.3 TRAINING ----------
ep_returns, losses, eps_hist, global_step = [], [], [], 0
t0 = time.time()
for ep in range(EPISODES):
    s, _ = env.reset(seed=SEED + ep); done, ep_ret = False, 0.0
    while not done:
        eps = max(EPS_END, EPS_START - (EPS_START - EPS_END) * global_step / EPS_DECAY_STEPS)   # linear ε decay
        if random.random() < eps: a = env.action_space.sample()                                   # explore
        else:
            with torch.no_grad(): a = int(q_net(torch.tensor(s, dtype=torch.float32)).argmax())   # exploit
        s2, r, term, trunc, _ = env.step(a); done = term or trunc
        buffer.push(s, a, r, s2, float(term))               # only `terminated` cuts the bootstrap (time-limit truncation does not)
        s, ep_ret, global_step = s2, ep_ret + r, global_step + 1

        if len(buffer) >= WARMUP:                           # learning step
            S, A, R, S2, D = buffer.sample(BATCH)
            q_sa = q_net(S).gather(1, A).squeeze(1)
            with torch.no_grad(): y = R + GAMMA * target_net(S2).max(1).values * (1 - D)     # target uses the frozen network
            loss = F.smooth_l1_loss(q_sa, y)
            optimizer.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(q_net.parameters(), 10.0); optimizer.step()
            losses.append(loss.item())
        if global_step % TARGET_SYNC == 0: target_net.load_state_dict(q_net.state_dict())      # sync target network
    ep_returns.append(ep_ret); eps_hist.append(eps)
    if (ep + 1) % 25 == 0:
        print(f"Episode {ep + 1:3d} | avg return (last 25): {np.mean(ep_returns[-25:]):6.1f} | ε={eps:.2f} | {time.time() - t0:.0f}s")

# ---------- 7.4 GREEDY EVALUATION ----------
eval_returns = []
for i in range(10):
    s, _ = env.reset(seed=1000 + i); done, tot = False, 0.0
    while not done:
        with torch.no_grad(): a = int(q_net(torch.tensor(s, dtype=torch.float32)).argmax())
        s, r, term, trunc, _ = env.step(a); done = term or trunc; tot += r
    eval_returns.append(tot)

# ---------- 7.5 VISUALIZATION ----------
fig, ax = plt.subplots(1, 4, figsize=(22, 4.2))
ax[0].plot(ep_returns, alpha=.35, label="episode return"); ax[0].plot(range(24, len(ep_returns)), smooth(ep_returns, 25), label="25-ep average")
ax[0].axhline(500, color="g", ls="--", label="max (500)"); ax[0].set(title="DQN training return", xlabel="Episode", ylabel="Return"); ax[0].legend()
ax[1].plot(smooth(losses, 200)); ax[1].set(title="TD loss (smoothed)", xlabel="Gradient step", ylabel="Huber loss")
ax[2].plot(eps_hist); ax[2].set(title="Exploration rate ε", xlabel="Episode", ylabel="ε")
ax[3].bar(range(10), eval_returns, color="tab:green"); ax[3].set(title="Greedy evaluation (10 episodes)", xlabel="Eval episode", ylabel="Return", ylim=(0, 520))
plt.tight_layout(); plt.show()

# ---------- 7.6 RESULTS ----------
print(f"Final greedy evaluation: mean return = {np.mean(eval_returns):.1f} ± {np.std(eval_returns):.1f} (max 500)")

# ---------- 7.7 FINAL SUMMARY ----------
import textwrap
print("\n" + "=" * 100 + "\nSUMMARY\n" + "=" * 100)
print(textwrap.fill(
    f"A Deep Q-Network with experience replay and a target network was trained on CartPole-v1 for {len(ep_returns)} episodes ({global_step} environment steps). "
    f"The best 25-episode average return during training was {smooth(ep_returns, 25).max():.1f} (maximum possible 500), and the final greedy evaluation over 10 fresh episodes averaged "
    f"{np.mean(eval_returns):.1f} ± {np.std(eval_returns):.1f}. The curve typically rises quickly once ε has decayed and the buffer has useful data, then may wobble or dip: DQN is known to be unstable because the "
    f"target moves as the network changes and one bad update can degrade the policy. The Huber loss plot shows the TD error shrinking and then fluctuating as returns grow (larger Q-values).", 100))

**Summary of Experiment 7.** WHAT WE DID: We trained a Deep Q-Network on CartPole-v1. A neural network replaces the Q-table, a replay
buffer provides decorrelated mini-batches, and a periodically-synchronised target network stabilises the
bootstrapped targets. The loss is the Huber loss between Q(s,a) and r + γ·max Q_target(s',a'). Exploration
uses an ε that decays linearly. HOW TO READ THE PLOTS: training return per episode (with moving average), the
loss, the ε schedule, and a greedy evaluation over 10 episodes. RESULT: the agent usually learns to balance
for several hundred steps within a few hundred episodes, but DQN performance can oscillate, so the final
number may differ between runs. See the printed summary.

## Experiment 8 — TD-Gammon Style Learning (TD(λ) + neural network + self-play)

**TD-Gammon** (Tesauro, 1992) learned world-class backgammon *only* by playing against itself. Its recipe:

1. A small neural network `V(s)` estimates the probability that White wins from board `s`.
2. The agent picks the move leading to the **after-state** with the best estimated value (White maximises, Black minimises); finished games use the true result.
3. After every move, update the weights with **TD(λ)** using eligibility traces:
   `δ = V(s') − V(s)`,  `e ← γλ·e + ∇_w V(s)`,  `w ← w + α·δ·e`
   (at the end of the game the target is the real result: 1 / 0.5 / 0).

⚠️ **Honest note:** full backgammon needs a few hundred lines of rule code and ~100k+ training games. To keep this notebook runnable in
minutes, we apply the **identical TD-Gammon algorithm to Tic-Tac-Toe** (X vs O). The network, eligibility traces, self-play and after-state
selection are exactly the TD-Gammon method; only the game is smaller. We track win-rate against a random opponent as training proceeds.

In [ ]:
# ============================================================
# EXPERIMENT 8: TD-GAMMON STYLE TD(λ) SELF-PLAY (TIC-TAC-TOE)
# ============================================================
# ---- STANDALONE SETUP (this cell needs nothing from any other cell) ----
import random, time, warnings                 # stdlib: randomness, timing, silence noisy warnings
import numpy as np                            # numerical arrays
import matplotlib.pyplot as plt               # all graphs
from collections import deque                 # replay buffers / fixed-length memories
warnings.filterwarnings("ignore")
SEED = 42                                     # fixed seed -> reproducible results
random.seed(SEED); np.random.seed(SEED)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})

# ---- SHARED HELPER: moving average to make noisy curves readable ----
def smooth(x, k=20):
    """Moving average used to make noisy reward curves readable."""
    x = np.asarray(x, dtype=float)
    if len(x) < k: return x
    return np.convolve(x, np.ones(k) / k, mode="valid")



# ---------- 8.1 GAME LOGIC ----------
LINES = [(0, 1, 2), (3, 4, 5), (6, 7, 8), (0, 3, 6), (1, 4, 7), (2, 5, 8), (0, 4, 8), (2, 4, 6)]


def winner(b):
    for i, j, k in LINES:
        if b[i] != 0 and b[i] == b[j] == b[k]: return b[i]
    return 0


def encode(b):
    """19 inputs like TD-Gammon's board encoding: 'X here' (9) + 'O here' (9) + 1 bit saying whose turn it is next.
    (Without the turn bit the net would have to learn piece-count parity, which is hard.)"""
    b = np.asarray(b); o_to_move = float((b == 1).sum() > (b == -1).sum())      # X just moved -> O to move
    return np.concatenate([(b == 1), (b == -1), [o_to_move]]).astype(float)


# ---------- 8.2 NEURAL NETWORK (NumPy, one hidden layer, sigmoid) ----------
class TDNet:
    def __init__(self, n_in=19, n_hid=40, seed=0):
        r = np.random.default_rng(seed)
        self.W1, self.b1 = r.normal(0, .3, (n_hid, n_in)), np.zeros(n_hid)
        self.W2, self.b2 = r.normal(0, .3, n_hid), 0.0
    @staticmethod
    def sig(z): return 1 / (1 + np.exp(-z))
    def value(self, X):                                           # X: (n, 19) batch -> P(X wins) for each row
        return self.sig(self.sig(X @ self.W1.T + self.b1) @ self.W2 + self.b2)
    def grads(self, x):                                           # gradient of V(x) w.r.t. all parameters
        h = self.sig(self.W1 @ x + self.b1); v = self.sig(h @ self.W2 + self.b2); dv = v * (1 - v)
        dh = dv * self.W2 * h * (1 - h)
        return [np.outer(dh, x), dh, dv * h, dv]                  # for W1, b1, W2, b2


def afterstates(b, player):
    moves = [i for i in range(9) if b[i] == 0]
    boards = []
    for m in moves:
        nb = list(b); nb[m] = player; boards.append(nb)
    return moves, boards


def choose(net, b, player, eps):
    moves, boards = afterstates(b, player)
    if random.random() < eps: i = random.randrange(len(moves))
    else:
        vals = net.value(np.array([encode(x) for x in boards]))   # evaluate ALL after-states in one batch
        for k, x in enumerate(boards):                            # game over after this move? use the true result, not the net
            w = winner(x)
            if w != 0 or 0 not in x: vals[k] = 1.0 if w == 1 else 0.0 if w == -1 else 0.5
        i = int(np.argmax(vals) if player == 1 else np.argmin(vals))  # X maximises P(X wins), O minimises
    return boards[i]


# ---------- 8.3 TD(λ) SELF-PLAY TRAINING ----------
def play_training_game(net, alpha, lam, eps, gamma=1.0):
    b, player = [0] * 9, 1
    e = [np.zeros_like(net.W1), np.zeros_like(net.b1), np.zeros_like(net.W2), 0.0]    # eligibility traces
    x_prev = None
    while True:
        b = choose(net, b, player, eps); x = encode(b); w = winner(b)
        done = w != 0 or 0 not in b
        v_next = (1.0 if w == 1 else 0.0 if w == -1 else 0.5) if done else float(net.value(x[None])[0])   # terminal -> true outcome
        if x_prev is not None:                                    # TD update for the PREVIOUS position
            g = net.grads(x_prev); delta = gamma * v_next - float(net.value(x_prev[None])[0])
            for i in range(4): e[i] = gamma * lam * e[i] + g[i]
            net.W1 += alpha * delta * e[0]; net.b1 += alpha * delta * e[1]; net.W2 += alpha * delta * e[2]; net.b2 += alpha * delta * e[3]
        if done:                                                  # final update toward the real game result
            g = net.grads(x); delta = v_next - float(net.value(x[None])[0])
            for i in range(4): e[i] = gamma * lam * e[i] + g[i]
            net.W1 += alpha * delta * e[0]; net.b1 += alpha * delta * e[1]; net.W2 += alpha * delta * e[2]; net.b2 += alpha * delta * e[3]
            return w
        x_prev, player = x, -player


def evaluate_vs_random(net, games=300):
    """Greedy agent vs uniformly random opponent, playing both sides. Returns (win, draw, loss) rates."""
    out = {}
    for side, name in [(1, "as X"), (-1, "as O")]:
        res = {"win": 0, "draw": 0, "loss": 0}
        for _ in range(games):
            b, player = [0] * 9, 1
            while True:
                if player == side: b = choose(net, b, player, 0.0)
                else: b[random.choice([i for i in range(9) if b[i] == 0])] = player
                w = winner(b)
                if w != 0 or 0 not in b: break
                player = -player
            res["win" if w == side else "loss" if w == -side else "draw"] += 1
        out[name] = {k: v / games for k, v in res.items()}
    return out


net = TDNet()
ALPHA, LAMBDA, EPS, TOTAL_GAMES, EVAL_EVERY = 0.05, 0.7, 0.1, 15_000, 1_500
curve, outcomes = [], []
r0 = evaluate_vs_random(net); curve.append((0, r0))
print("Before training:", {k: {m: round(x, 2) for m, x in v.items()} for k, v in r0.items()})

# ---------- 8.4 TRAINING ----------
t0 = time.time()
for g in range(1, TOTAL_GAMES + 1):
    alpha_g = ALPHA * max(0.1, 1 - g / TOTAL_GAMES)                  # linearly decay the learning rate
    outcomes.append(play_training_game(net, alpha_g, LAMBDA, EPS))
    if g % EVAL_EVERY == 0:
        res = evaluate_vs_random(net); curve.append((g, res))
        print(f"Game {g:6d} | as X: win {res['as X']['win']:.2f} loss {res['as X']['loss']:.2f} | as O: win {res['as O']['win']:.2f} loss {res['as O']['loss']:.2f} | {time.time() - t0:.0f}s")

# ---------- 8.5 VISUALIZATION ----------
games = [c[0] for c in curve]
fig, ax = plt.subplots(1, 3, figsize=(20, 4.3))
for side, col in [("as X", "tab:blue"), ("as O", "tab:red")]:
    ax[0].plot(games, [c[1][side]["win"] for c in curve], "-o", color=col, label=f"win rate {side}")
    ax[0].plot(games, [c[1][side]["loss"] for c in curve], "--x", color=col, alpha=.6, label=f"loss rate {side}")
ax[0].set(title="Learning curve vs random opponent", xlabel="Self-play games", ylabel="Rate"); ax[0].legend(fontsize=8)
o = np.array(outcomes); w = 500
ax[1].plot(smooth(o == 1, w), label="X wins"); ax[1].plot(smooth(o == -1, w), label="O wins"); ax[1].plot(smooth(o == 0, w), label="Draws")
ax[1].set(title="Self-play outcomes (rolling 500 games)", xlabel="Self-play game", ylabel="Fraction"); ax[1].legend()
empty = [0] * 9; moves, boards = afterstates(empty, 1); vals = net.value(np.array([encode(x) for x in boards]))
ax[2].imshow(np.array(vals).reshape(3, 3), cmap="viridis"); ax[2].grid(False); ax[2].set_title("Learned P(X wins) after X's first move")
for m, v in zip(moves, vals): ax[2].text(m % 3, m // 3, f"{v:.2f}", ha="center", va="center", color="w", fontsize=12)
ax[2].set_xticks([]); ax[2].set_yticks([])
plt.tight_layout(); plt.show()

# ---------- 8.6 RESULTS ----------
final = curve[-1][1]
print("Final vs random ->", {k: {m: round(x, 2) for m, x in v.items()} for k, v in final.items()})
print("A random opponent is beatable, but tic-tac-toe is a draw with perfect play, so ~95% wins as X (first player) is near the practical ceiling;")
print("as O (second player) the best possible vs a random X is lower because X often gets a head start.")

# ---------- 8.7 FINAL SUMMARY ----------
import textwrap
print("\n" + "=" * 100 + "\nSUMMARY\n" + "=" * 100)
print(textwrap.fill(
    f"A one-hidden-layer neural network was trained by TD(λ={LAMBDA}, α={ALPHA}) from {TOTAL_GAMES} self-play Tic-Tac-Toe games using the TD-Gammon recipe (after-state evaluation, eligibility traces, "
    f"a turn indicator input). Against a uniformly random opponent the greedy agent's win rate went from {curve[0][1]['as X']['win']:.2f} (as X) and {curve[0][1]['as O']['win']:.2f} (as O) before training "
    f"to {final['as X']['win']:.2f} and {final['as O']['win']:.2f} after training, with losses of {final['as X']['loss']:.2f} and {final['as O']['loss']:.2f}. No move was ever labelled by a teacher: the only supervision was the final "
    f"win/draw/loss signal propagated backwards through time by the traces. Results fluctuate a little between evaluation points because self-play is non-stationary.", 100))

**Summary of Experiment 8.** WHAT WE DID: We reproduced the TD-Gammon learning method: a neural network estimates the probability that X
wins from any board, moves are chosen by looking one move ahead at the after-states, and the weights are
updated after every move with TD(λ) using eligibility traces; the only real reward is the final game result.
Because full backgammon is large, the game here is Tic-Tac-Toe, which keeps the exact algorithm but trains in
seconds. HOW TO READ THE PLOTS: win/loss rate against a random player at checkpoints during training, rolling
self-play outcomes, and the learned value of X's first-move squares. RESULT: the agent improves from near-
random play to winning most games against a random opponent without any hand-written strategy. See the printed
summary.

## Experiment 9 — CartPole with REINFORCE (Monte Carlo Policy Gradient)

Instead of learning values, **learn the policy directly**: a network outputs action probabilities `π_θ(a|s)`.

**Policy-gradient theorem:** `∇J(θ) = E[ Σ_t ∇ log π_θ(a_t|s_t) · G_t ]` — increase the probability of actions that were followed by high returns.

**REINFORCE algorithm:** play a full episode → compute discounted returns `G_t` → normalise them (acts as a simple baseline, reduces variance)
→ gradient step on `−Σ log π(a_t|s_t)·G_t`. Because it uses whole-episode Monte Carlo returns it is **unbiased but high-variance**,
so the learning curve is noisy.

In [ ]:
# ============================================================
# EXPERIMENT 9: REINFORCE ON CARTPOLE
# ============================================================
# ---- STANDALONE SETUP (this cell needs nothing from any other cell) ----
import random, time, warnings                 # stdlib: randomness, timing, silence noisy warnings
import numpy as np                            # numerical arrays
import matplotlib.pyplot as plt               # all graphs
from collections import deque                 # replay buffers / fixed-length memories
warnings.filterwarnings("ignore")
SEED = 42                                     # fixed seed -> reproducible results
random.seed(SEED); np.random.seed(SEED)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})
import gymnasium as gym                       # CartPole environment
import torch                                  # neural networks + autograd
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical   # sampling actions from a policy
torch.set_num_threads(1)                      # tiny networks run fastest on 1 thread
torch.manual_seed(SEED)

# ---- SHARED HELPER: moving average to make noisy curves readable ----
def smooth(x, k=20):
    """Moving average used to make noisy reward curves readable."""
    x = np.asarray(x, dtype=float)
    if len(x) < k: return x
    return np.convolve(x, np.ones(k) / k, mode="valid")



# ---------- 9.1 HYPERPARAMETERS ----------
GAMMA, LR, EPISODES = 0.99, 3e-3, 600
torch.manual_seed(SEED); np.random.seed(SEED)


# ---------- 9.2 POLICY NETWORK ----------
class PolicyNet(nn.Module):
    def __init__(self, n_obs=4, n_act=2):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_obs, 128), nn.ReLU(), nn.Linear(128, n_act))
    def forward(self, x): return Categorical(logits=self.net(x))        # returns a distribution over actions


env = gym.make("CartPole-v1")
policy = PolicyNet(); optimizer = optim.Adam(policy.parameters(), lr=LR)

# ---------- 9.3 TRAINING ----------
ep_returns, ep_losses, entropies = [], [], []
t0 = time.time()
for ep in range(EPISODES):
    s, _ = env.reset(seed=SEED + ep); log_probs, rewards, ents, done = [], [], [], False
    while not done:                                                      # --- collect one full episode
        dist = policy(torch.tensor(s, dtype=torch.float32)); a = dist.sample()
        log_probs.append(dist.log_prob(a)); ents.append(dist.entropy())
        s, r, term, trunc, _ = env.step(a.item()); rewards.append(r); done = term or trunc
    G, returns = 0.0, []                                                 # --- discounted returns, computed backwards
    for r in reversed(rewards): G = r + GAMMA * G; returns.insert(0, G)
    returns = torch.tensor(returns, dtype=torch.float32)
    returns = (returns - returns.mean()) / (returns.std() + 1e-8)        # normalise = variance reduction
    loss = -(torch.stack(log_probs) * returns).sum()                     # --- REINFORCE loss
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    ep_returns.append(sum(rewards)); ep_losses.append(loss.item()); entropies.append(torch.stack(ents).mean().item())
    if (ep + 1) % 50 == 0:
        print(f"Episode {ep + 1:3d} | avg return (last 50): {np.mean(ep_returns[-50:]):6.1f} | {time.time() - t0:.0f}s")

# ---------- 9.4 EVALUATION ----------
eval_returns = []
for i in range(10):
    s, _ = env.reset(seed=2000 + i); done, tot = False, 0.0
    while not done:
        with torch.no_grad(): a = policy.net(torch.tensor(s, dtype=torch.float32)).argmax().item()      # greedy action
        s, r, term, trunc, _ = env.step(a); tot += r; done = term or trunc
    eval_returns.append(tot)

# ---------- 9.5 VISUALIZATION ----------
fig, ax = plt.subplots(1, 4, figsize=(22, 4.2))
ax[0].plot(ep_returns, alpha=.35); ax[0].plot(range(19, len(ep_returns)), smooth(ep_returns, 20), label="20-ep average")
ax[0].axhline(500, color="g", ls="--"); ax[0].set(title="REINFORCE training return", xlabel="Episode", ylabel="Return"); ax[0].legend()
ax[1].plot(smooth(ep_losses, 20)); ax[1].set(title="Policy-gradient loss (smoothed)", xlabel="Episode", ylabel="Loss")
ax[2].plot(smooth(entropies, 20)); ax[2].set(title="Policy entropy (exploration)", xlabel="Episode", ylabel="Entropy (max ln2≈0.69)")
ax[3].bar(range(10), eval_returns, color="tab:green"); ax[3].set(title="Greedy evaluation", xlabel="Eval episode", ylabel="Return", ylim=(0, 520))
plt.tight_layout(); plt.show()

# ---------- 9.6 RESULTS ----------
print(f"Best 20-episode average during training: {smooth(ep_returns, 20).max():.1f}")
print(f"Final greedy evaluation: {np.mean(eval_returns):.1f} ± {np.std(eval_returns):.1f}")

# ---------- 9.7 FINAL SUMMARY ----------
import textwrap
print("\n" + "=" * 100 + "\nSUMMARY\n" + "=" * 100)
print(textwrap.fill(
    f"REINFORCE learned a CartPole policy directly (no value function) over {len(ep_returns)} episodes using Monte Carlo returns normalised to reduce variance. "
    f"The best 20-episode average training return was {smooth(ep_returns, 20).max():.1f} and the greedy evaluation over 10 episodes averaged {np.mean(eval_returns):.1f} ± {np.std(eval_returns):.1f}. "
    f"Because whole-episode returns are very noisy, the training curve is jagged and can even drop after reaching the maximum; the entropy plot shows the policy becoming more deterministic as it improves. "
    f"A smaller learning rate (3e-3) was used because larger steps made the policy collapse in testing.", 100))

**Summary of Experiment 9.** WHAT WE DID: We trained a policy network on CartPole with REINFORCE, the simplest policy-gradient method. Each
episode is played to the end, discounted returns are computed, normalised, and the loss -Σ log π(a|s)·G is
minimised, which makes actions followed by high returns more likely. HOW TO READ THE PLOTS: training return,
loss, policy entropy (how random the policy still is), and a greedy evaluation. RESULT: REINFORCE can reach
the maximum score of 500 but the curve is noisy because Monte Carlo returns have high variance; the printed
summary has the numbers for this run.

## Experiment 10 — Actor-Critic (one-step, online)

REINFORCE waits until the episode ends. **Actor-Critic** adds a second network — the **critic** `V_w(s)` — so we can learn *every step*:

- **Critic** (value): minimise the TD error `δ = r + γ V(s') − V(s)` (squared).
- **Actor** (policy): `∇θ ∝ ∇ log π(a|s) · δ` — δ is a low-variance estimate of the *advantage* (was this action better than expected?).

Trade-off vs REINFORCE: much lower variance, but a little bias because the critic is imperfect (especially early on).

In [ ]:
# ============================================================
# EXPERIMENT 10: ONE-STEP ACTOR-CRITIC (CARTPOLE)
# ============================================================
# ---- STANDALONE SETUP (this cell needs nothing from any other cell) ----
import random, time, warnings                 # stdlib: randomness, timing, silence noisy warnings
import numpy as np                            # numerical arrays
import matplotlib.pyplot as plt               # all graphs
from collections import deque                 # replay buffers / fixed-length memories
warnings.filterwarnings("ignore")
SEED = 42                                     # fixed seed -> reproducible results
random.seed(SEED); np.random.seed(SEED)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})
import gymnasium as gym                       # CartPole environment
import torch                                  # neural networks + autograd
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical   # sampling actions from a policy
torch.set_num_threads(1)                      # tiny networks run fastest on 1 thread
torch.manual_seed(SEED)

# ---- SHARED HELPER: moving average to make noisy curves readable ----
def smooth(x, k=20):
    """Moving average used to make noisy reward curves readable."""
    x = np.asarray(x, dtype=float)
    if len(x) < k: return x
    return np.convolve(x, np.ones(k) / k, mode="valid")



# ---------- 10.1 HYPERPARAMETERS ----------
GAMMA, LR_ACTOR, LR_CRITIC, EPISODES, ENT_COEF = 0.99, 1e-3, 5e-3, 400, 0.01
torch.manual_seed(SEED); random.seed(SEED)

# ---------- 10.2 ACTOR + CRITIC NETWORKS ----------
env = gym.make("CartPole-v1")
actor = nn.Sequential(nn.Linear(4, 128), nn.ReLU(), nn.Linear(128, 2))        # outputs action logits
critic = nn.Sequential(nn.Linear(4, 128), nn.ReLU(), nn.Linear(128, 1))       # outputs V(s)
opt_a, opt_c = optim.Adam(actor.parameters(), lr=LR_ACTOR), optim.Adam(critic.parameters(), lr=LR_CRITIC)

# ---------- 10.3 TRAINING ----------
ep_returns, critic_losses, actor_losses = [], [], []
import copy; best_avg, best_actor, recent = -1.0, None, deque(maxlen=10)   # one-step AC can be unstable: keep the best-performing actor
t0 = time.time()
for ep in range(EPISODES):
    s, _ = env.reset(seed=SEED + ep); done, ep_ret, cl, al, n = False, 0.0, 0.0, 0.0, 0
    while not done:
        st = torch.tensor(s, dtype=torch.float32)
        dist = Categorical(logits=actor(st)); a = dist.sample()
        s2, r, term, trunc, _ = env.step(a.item()); done = term or trunc
        s2t = torch.tensor(s2, dtype=torch.float32)

        v = critic(st).squeeze()
        with torch.no_grad(): v_next = critic(s2t).squeeze() * (1.0 - float(term))     # no bootstrap at a true terminal state
        td_error = r + GAMMA * v_next - v                                              # δ = r + γV(s') − V(s)

        critic_loss = td_error.pow(2)                                                  # critic learns V by minimising δ²
        opt_c.zero_grad(); critic_loss.backward(); opt_c.step()

        actor_loss = -dist.log_prob(a) * td_error.detach() - ENT_COEF * dist.entropy()  # actor uses δ as the advantage
        opt_a.zero_grad(); actor_loss.backward(); opt_a.step()

        s, ep_ret, cl, al, n = s2, ep_ret + r, cl + critic_loss.item(), al + actor_loss.item(), n + 1
    ep_returns.append(ep_ret); critic_losses.append(cl / n); actor_losses.append(al / n)
    if len(ep_returns) >= 20 and np.mean(ep_returns[-20:]) > best_avg and len(recent) == 10: best_avg, best_actor = np.mean(ep_returns[-20:]), recent[0]   # weights from mid-window (before any collapse)
    recent.append(copy.deepcopy(actor.state_dict()))
    if (ep + 1) % 50 == 0:
        print(f"Episode {ep + 1:3d} | avg return (last 50): {np.mean(ep_returns[-50:]):6.1f} | {time.time() - t0:.0f}s")

# ---------- 10.4 EVALUATION (using the best checkpoint) ----------
print(f"Restoring best actor checkpoint (20-episode avg return {best_avg:.1f}); the final weights may have drifted.")
actor.load_state_dict(best_actor)
eval_returns = []
for i in range(10):
    s, _ = env.reset(seed=3000 + i); done, tot = False, 0.0
    while not done:
        with torch.no_grad(): a = Categorical(logits=actor(torch.tensor(s, dtype=torch.float32))).sample().item()   # sample from the stochastic policy (as in training)
        s, r, term, trunc, _ = env.step(a); tot += r; done = term or trunc
    eval_returns.append(tot)

# ---------- 10.5 VISUALIZATION ----------
fig, ax = plt.subplots(1, 4, figsize=(22, 4.2))
ax[0].plot(ep_returns, alpha=.35); ax[0].plot(range(19, len(ep_returns)), smooth(ep_returns, 20), label="20-ep average")
ax[0].axhline(500, color="g", ls="--"); ax[0].set(title="Actor-Critic training return", xlabel="Episode", ylabel="Return"); ax[0].legend()
ax[1].plot(smooth(critic_losses, 20)); ax[1].set(title="Critic loss (mean δ²)", xlabel="Episode", ylabel="Loss")
ax[2].plot(smooth(actor_losses, 20)); ax[2].set(title="Actor loss (mean)", xlabel="Episode", ylabel="Loss")
ax[3].bar(range(10), eval_returns, color="tab:green"); ax[3].set(title="Evaluation (sampled policy)", xlabel="Eval episode", ylabel="Return", ylim=(0, 520))
plt.tight_layout(); plt.show()

# ---------- 10.6 RESULTS ----------
print(f"Best 20-episode average during training: {smooth(ep_returns, 20).max():.1f}")
print(f"Evaluation of best checkpoint (sampled actions): {np.mean(eval_returns):.1f} ± {np.std(eval_returns):.1f}")

# ---------- 10.7 FINAL SUMMARY ----------
import textwrap
print("\n" + "=" * 100 + "\nSUMMARY\n" + "=" * 100)
print(textwrap.fill(
    f"The one-step Actor-Critic updated an actor (policy) and a critic (value) after every single transition using the TD error as the advantage. Over {len(ep_returns)} episodes the best 20-episode training average was "
    f"{best_avg:.1f}; evaluating the saved best checkpoint with sampled actions gave {np.mean(eval_returns):.1f} ± {np.std(eval_returns):.1f}. This method learns faster per episode than REINFORCE (updates every step, lower variance) "
    f"but is less stable: with plain per-step SGD the policy can collapse late in training, which is why we keep the best checkpoint. Evaluation samples from the stochastic policy because that is the policy that was trained.", 100))

**Summary of Experiment 10.** WHAT WE DID: We added a critic to the policy-gradient idea. After every single step, the critic is trained to
reduce the TD error δ = r + γV(s') - V(s), and the actor is pushed in the direction of ∇log π(a|s)·δ, so the
TD error acts as an advantage estimate. We keep the best-performing actor snapshot because this simple online
version can become unstable. HOW TO READ THE PLOTS: training return, critic loss, actor loss and an evaluation
of the saved checkpoint. RESULT: learning starts earlier than REINFORCE because updates happen every step, but
stability is weaker, which motivates A2C and PPO in the next two experiments.

## Experiment 11 — Actor-Critic Methods: Advantage Actor-Critic (A2C)

A2C is the *batched, n-step* version of actor-critic and the synchronous sibling of **A3C** (A3C runs the workers asynchronously in threads).

**Ingredients**
- **Parallel environments** (here 8) → decorrelated data without a replay buffer.
- **n-step returns** (n=5): `R_t = r_t + γ r_{t+1} + … + γ^n V(s_{t+n})` and advantage `A_t = R_t − V(s_t)`.
- **Shared network** with two heads (policy logits + value).
- **Loss** = policy loss `−log π · A` + `0.5 ·` value loss + `−0.01 ·` entropy bonus (keeps exploring).

In [ ]:
# ============================================================
# EXPERIMENT 11: ADVANTAGE ACTOR-CRITIC (A2C) — CARTPOLE
# ============================================================
# ---- STANDALONE SETUP (this cell needs nothing from any other cell) ----
import random, time, warnings                 # stdlib: randomness, timing, silence noisy warnings
import numpy as np                            # numerical arrays
import matplotlib.pyplot as plt               # all graphs
from collections import deque                 # replay buffers / fixed-length memories
warnings.filterwarnings("ignore")
SEED = 42                                     # fixed seed -> reproducible results
random.seed(SEED); np.random.seed(SEED)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})
import gymnasium as gym                       # CartPole environment
import torch                                  # neural networks + autograd
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical   # sampling actions from a policy
torch.set_num_threads(1)                      # tiny networks run fastest on 1 thread
torch.manual_seed(SEED)

# ---- SHARED HELPER: moving average to make noisy curves readable ----
def smooth(x, k=20):
    """Moving average used to make noisy reward curves readable."""
    x = np.asarray(x, dtype=float)
    if len(x) < k: return x
    return np.convolve(x, np.ones(k) / k, mode="valid")

# ---- HELPER: vectorised CartPole (n copies, auto-reset) and MLP builder ----
# ---------- tiny vectorised CartPole (used by A2C and PPO) ----------
class SimpleVecEnv:
    """Runs n CartPole copies in lock-step with auto-reset; logs (global_step, episode_return)."""
    def __init__(self, n, seed=0):
        self.n = n
        self.envs = [gym.make("CartPole-v1") for _ in range(n)]
        self.obs = np.array([e.reset(seed=seed + i)[0] for i, e in enumerate(self.envs)], dtype=np.float32)
        self.ep_ret, self.t, self.finished = np.zeros(n), 0, []

    def step(self, actions):
        rews, dones = np.zeros(self.n, dtype=np.float32), np.zeros(self.n, dtype=np.float32)
        for i, (e, a) in enumerate(zip(self.envs, actions)):
            o, r, term, trunc, _ = e.step(int(a))
            self.ep_ret[i] += r; self.t += 1
            if term or trunc:
                self.finished.append((self.t, self.ep_ret[i])); self.ep_ret[i] = 0; o, _ = e.reset(); dones[i] = 1.0
            self.obs[i], rews[i] = o, r
        return self.obs.copy(), rews, dones


def mlp(inp, out, hidden=64):
    return nn.Sequential(nn.Linear(inp, hidden), nn.Tanh(), nn.Linear(hidden, hidden), nn.Tanh(), nn.Linear(hidden, out))



# ---------- 11.1 HYPERPARAMETERS ----------
N_ENVS, N_STEPS, TOTAL_STEPS = 8, 5, 150_000
GAMMA, LR, VF_COEF, ENT_COEF = 0.99, 1e-3, 0.5, 0.01
torch.manual_seed(SEED)


# ---------- 11.2 SHARED ACTOR-CRITIC NETWORK ----------
class ActorCritic(nn.Module):
    def __init__(self, n_obs=4, n_act=2, hid=64):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(n_obs, hid), nn.Tanh(), nn.Linear(hid, hid), nn.Tanh())
        self.pi, self.v = nn.Linear(hid, n_act), nn.Linear(hid, 1)
    def forward(self, x):
        h = self.body(x); return Categorical(logits=self.pi(h)), self.v(h).squeeze(-1)


model = ActorCritic(); optimizer = optim.Adam(model.parameters(), lr=LR)
venv = SimpleVecEnv(N_ENVS, seed=SEED)
obs = torch.tensor(venv.obs)
pl_hist, vl_hist, ent_hist = [], [], []

# ---------- 11.3 TRAINING ----------
t0, n_updates = time.time(), TOTAL_STEPS // (N_ENVS * N_STEPS)
for upd in range(n_updates):
    logps, values, rewards, dones, ents = [], [], [], [], []
    for _ in range(N_STEPS):                                             # --- roll out n steps in every env
        dist, v = model(obs); a = dist.sample()
        o, r, d = venv.step(a.numpy())
        logps.append(dist.log_prob(a)); values.append(v); ents.append(dist.entropy())
        rewards.append(torch.tensor(r)); dones.append(torch.tensor(d)); obs = torch.tensor(o)
    with torch.no_grad(): _, next_v = model(obs)                         # bootstrap value after the last step
    R, returns = next_v, []
    for t in reversed(range(N_STEPS)):                                   # --- n-step returns (reset at episode ends)
        R = rewards[t] + GAMMA * R * (1 - dones[t]); returns.insert(0, R)
    returns, values, logps = torch.stack(returns), torch.stack(values), torch.stack(logps)
    adv = returns - values                                               # advantage = n-step return − V(s)
    policy_loss = -(logps * adv.detach()).mean()                         # actor
    value_loss = adv.pow(2).mean()                                       # critic
    entropy = torch.stack(ents).mean()
    loss = policy_loss + VF_COEF * value_loss - ENT_COEF * entropy
    optimizer.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 0.5); optimizer.step()
    pl_hist.append(policy_loss.item()); vl_hist.append(value_loss.item()); ent_hist.append(entropy.item())
    if (upd + 1) % 500 == 0 and venv.finished:
        print(f"Step {venv.t:7d} | avg return (last 20 eps): {np.mean([x[1] for x in venv.finished[-20:]]):6.1f} | {time.time() - t0:.0f}s")

# ---------- 11.4 VISUALIZATION ----------
steps_, rets_ = np.array([x[0] for x in venv.finished]), np.array([x[1] for x in venv.finished])
fig, ax = plt.subplots(1, 4, figsize=(22, 4.2))
ax[0].plot(steps_, rets_, alpha=.25); ax[0].plot(steps_[29:], smooth(rets_, 30), label="30-ep average")
ax[0].axhline(500, color="g", ls="--"); ax[0].set(title="A2C training return", xlabel="Environment steps", ylabel="Return"); ax[0].legend()
ax[1].plot(smooth(pl_hist, 100)); ax[1].set(title="Policy loss", xlabel="Update", ylabel="Loss")
ax[2].plot(smooth(vl_hist, 100)); ax[2].set(title="Value loss", xlabel="Update", ylabel="MSE")
ax[3].plot(smooth(ent_hist, 100)); ax[3].set(title="Policy entropy", xlabel="Update", ylabel="Entropy")
plt.tight_layout(); plt.show()

# ---------- 11.5 RESULTS ----------
print(f"Episodes completed: {len(rets_)} | best 30-episode average: {smooth(rets_, 30).max():.1f} | last 30 episodes avg: {rets_[-30:].mean():.1f}")

# ---------- 11.6 FINAL SUMMARY ----------
import textwrap
print("\n" + "=" * 100 + "\nSUMMARY\n" + "=" * 100)
print(textwrap.fill(
    f"A2C used {N_ENVS} parallel CartPole environments, {N_STEPS}-step returns, a shared actor-critic network and an entropy bonus, trained for {venv.t} environment steps ({len(rets_)} finished episodes). "
    f"The best 30-episode average return was {smooth(rets_, 30).max():.1f} and the last 30 episodes averaged {rets_[-30:].mean():.1f} (max 500). Parallel environments decorrelate the data without a replay buffer and the n-step advantage "
    f"gives lower variance than REINFORCE, but there is no mechanism to limit step size, so progress is noisier than PPO. The value loss shows the critic adapting as returns grow; entropy falls as the policy becomes confident.", 100))

**Summary of Experiment 11.** WHAT WE DID: We implemented Advantage Actor-Critic (A2C), the synchronous version of A3C. Eight CartPole
copies run in parallel, every update uses 5-step returns bootstrapped from the critic, the actor and critic
share a network body, and an entropy bonus keeps exploration alive. HOW TO READ THE PLOTS: training return
against environment steps, the policy loss, the value loss and the entropy. RESULT: learning is faster and
lower-variance than REINFORCE and needs no replay buffer, but progress is still somewhat noisy because nothing
limits how far a single update can move the policy. See the printed summary for the measured returns.

## Experiment 12 — Proximal Policy Optimization (PPO)

Vanilla policy gradients can take a destructively large step. **PPO** keeps the new policy close to the old one using a **clipped surrogate objective**:

`L_clip = E[ min( ρ_t · A_t ,  clip(ρ_t, 1−ε, 1+ε) · A_t ) ]`, with the probability ratio `ρ_t = π_new(a|s) / π_old(a|s)`.

**Recipe:**
1. Collect a batch of rollouts with the current (old) policy.
2. Compute advantages with **GAE(λ)** (a smooth blend between 1-step and Monte Carlo advantages).
3. Re-use the same batch for several **epochs of mini-batch SGD** — safe because of the clip.
4. Total loss = −L_clip + 0.5·value loss − 0.01·entropy. We also monitor the **clip fraction** and **approx. KL**.

In [ ]:
# ============================================================
# EXPERIMENT 12: PROXIMAL POLICY OPTIMIZATION (PPO) — CARTPOLE
# ============================================================
# ---- STANDALONE SETUP (this cell needs nothing from any other cell) ----
import random, time, warnings                 # stdlib: randomness, timing, silence noisy warnings
import numpy as np                            # numerical arrays
import matplotlib.pyplot as plt               # all graphs
from collections import deque                 # replay buffers / fixed-length memories
warnings.filterwarnings("ignore")
SEED = 42                                     # fixed seed -> reproducible results
random.seed(SEED); np.random.seed(SEED)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})
import gymnasium as gym                       # CartPole environment
import torch                                  # neural networks + autograd
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical   # sampling actions from a policy
torch.set_num_threads(1)                      # tiny networks run fastest on 1 thread
torch.manual_seed(SEED)

# ---- SHARED HELPER: moving average to make noisy curves readable ----
def smooth(x, k=20):
    """Moving average used to make noisy reward curves readable."""
    x = np.asarray(x, dtype=float)
    if len(x) < k: return x
    return np.convolve(x, np.ones(k) / k, mode="valid")

# ---- HELPER: vectorised CartPole (n copies, auto-reset) and MLP builder ----
# ---------- tiny vectorised CartPole (used by A2C and PPO) ----------
class SimpleVecEnv:
    """Runs n CartPole copies in lock-step with auto-reset; logs (global_step, episode_return)."""
    def __init__(self, n, seed=0):
        self.n = n
        self.envs = [gym.make("CartPole-v1") for _ in range(n)]
        self.obs = np.array([e.reset(seed=seed + i)[0] for i, e in enumerate(self.envs)], dtype=np.float32)
        self.ep_ret, self.t, self.finished = np.zeros(n), 0, []

    def step(self, actions):
        rews, dones = np.zeros(self.n, dtype=np.float32), np.zeros(self.n, dtype=np.float32)
        for i, (e, a) in enumerate(zip(self.envs, actions)):
            o, r, term, trunc, _ = e.step(int(a))
            self.ep_ret[i] += r; self.t += 1
            if term or trunc:
                self.finished.append((self.t, self.ep_ret[i])); self.ep_ret[i] = 0; o, _ = e.reset(); dones[i] = 1.0
            self.obs[i], rews[i] = o, r
        return self.obs.copy(), rews, dones


def mlp(inp, out, hidden=64):
    return nn.Sequential(nn.Linear(inp, hidden), nn.Tanh(), nn.Linear(hidden, hidden), nn.Tanh(), nn.Linear(hidden, out))



# ---------- 12.1 HYPERPARAMETERS ----------
N_ENVS, N_STEPS, TOTAL_STEPS = 8, 128, 200_000
GAMMA, GAE_LAMBDA, CLIP, EPOCHS, MINIBATCHES = 0.99, 0.95, 0.2, 4, 4
LR, VF_COEF, ENT_COEF, MAX_GRAD = 2.5e-4 * 3, 0.5, 0.01, 0.5
torch.manual_seed(SEED)

# ---------- 12.2 NETWORKS (separate actor and critic, as in standard PPO) ----------
actor, critic = mlp(4, 2), mlp(4, 1)
optimizer = optim.Adam(list(actor.parameters()) + list(critic.parameters()), lr=LR, eps=1e-5)
venv = SimpleVecEnv(N_ENVS, seed=SEED)
obs = torch.tensor(venv.obs)
n_updates = TOTAL_STEPS // (N_ENVS * N_STEPS)
logs = {"pg": [], "vf": [], "ent": [], "clipfrac": [], "kl": []}

# ---------- 12.3 TRAINING ----------
t0 = time.time()
for upd in range(n_updates):
    for g in optimizer.param_groups: g["lr"] = LR * (1 - upd / n_updates)             # linear learning-rate annealing

    # ----- (a) collect rollout with the current policy -----
    b_obs, b_act, b_logp, b_rew, b_done, b_val = [], [], [], [], [], []
    for _ in range(N_STEPS):
        with torch.no_grad():
            dist = Categorical(logits=actor(obs)); a = dist.sample(); v = critic(obs).squeeze(-1)
        o, r, d = venv.step(a.numpy())
        b_obs.append(obs); b_act.append(a); b_logp.append(dist.log_prob(a)); b_val.append(v)
        b_rew.append(torch.tensor(r)); b_done.append(torch.tensor(d)); obs = torch.tensor(o)

    # ----- (b) GAE advantages -----
    with torch.no_grad(): next_v = critic(obs).squeeze(-1)
    adv, last = torch.zeros(N_STEPS, N_ENVS), torch.zeros(N_ENVS)
    for t in reversed(range(N_STEPS)):
        nv = next_v if t == N_STEPS - 1 else b_val[t + 1]
        delta = b_rew[t] + GAMMA * nv * (1 - b_done[t]) - b_val[t]                   # one-step TD error
        last = delta + GAMMA * GAE_LAMBDA * (1 - b_done[t]) * last                   # exponentially-weighted sum of TD errors
        adv[t] = last
    ret = adv + torch.stack(b_val)                                                    # targets for the critic
    obs_f, act_f, logp_old = torch.stack(b_obs).reshape(-1, 4), torch.stack(b_act).reshape(-1), torch.stack(b_logp).reshape(-1)
    adv_f, ret_f = adv.reshape(-1), ret.reshape(-1)

    # ----- (c) several epochs of clipped-objective mini-batch updates -----
    idx_all, mb = np.arange(N_ENVS * N_STEPS), (N_ENVS * N_STEPS) // MINIBATCHES
    for _ in range(EPOCHS):
        np.random.shuffle(idx_all)
        for k in range(MINIBATCHES):
            idx = idx_all[k * mb:(k + 1) * mb]
            dist = Categorical(logits=actor(obs_f[idx])); logp = dist.log_prob(act_f[idx])
            ratio = (logp - logp_old[idx]).exp()                                      # ρ = π_new / π_old
            A = (adv_f[idx] - adv_f[idx].mean()) / (adv_f[idx].std() + 1e-8)         # normalise advantages per mini-batch
            pg_loss = -torch.min(ratio * A, torch.clamp(ratio, 1 - CLIP, 1 + CLIP) * A).mean()   # clipped surrogate
            v_loss = 0.5 * (critic(obs_f[idx]).squeeze(-1) - ret_f[idx]).pow(2).mean()
            ent = dist.entropy().mean()
            loss = pg_loss + VF_COEF * v_loss - ENT_COEF * ent
            optimizer.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(list(actor.parameters()) + list(critic.parameters()), MAX_GRAD); optimizer.step()
    with torch.no_grad():
        logs["pg"].append(pg_loss.item()); logs["vf"].append(v_loss.item()); logs["ent"].append(ent.item())
        logs["clipfrac"].append(((ratio - 1).abs() > CLIP).float().mean().item()); logs["kl"].append(((ratio - 1) - (ratio.log())).mean().item())
    if (upd + 1) % 25 == 0 and venv.finished:
        print(f"Update {upd + 1:3d}/{n_updates} | step {venv.t:7d} | avg return (last 20 eps): {np.mean([x[1] for x in venv.finished[-20:]]):6.1f} | {time.time() - t0:.0f}s")

# ---------- 12.4 EVALUATION ----------
env = gym.make("CartPole-v1"); eval_returns = []
for i in range(10):
    s, _ = env.reset(seed=4000 + i); done, tot = False, 0.0
    while not done:
        with torch.no_grad(): a = actor(torch.tensor(s, dtype=torch.float32)).argmax().item()
        s, r, term, trunc, _ = env.step(a); tot += r; done = term or trunc
    eval_returns.append(tot)

# ---------- 12.5 VISUALIZATION ----------
steps_, rets_ = np.array([x[0] for x in venv.finished]), np.array([x[1] for x in venv.finished])
fig, ax = plt.subplots(1, 5, figsize=(26, 4.2))
ax[0].plot(steps_, rets_, alpha=.25); ax[0].plot(steps_[29:], smooth(rets_, 30), label="30-ep average")
ax[0].axhline(500, color="g", ls="--"); ax[0].set(title="PPO training return", xlabel="Environment steps", ylabel="Return"); ax[0].legend()
ax[1].plot(logs["pg"]); ax[1].set(title="Policy (clipped) loss", xlabel="Update")
ax[2].plot(logs["vf"]); ax[2].set(title="Value loss", xlabel="Update"); ax[2].set_yscale("log")
ax[3].plot(logs["clipfrac"], label="clip fraction"); ax[3].plot(logs["kl"], label="approx KL"); ax[3].set(title="PPO trust-region diagnostics", xlabel="Update"); ax[3].legend()
ax[4].bar(range(10), eval_returns, color="tab:green"); ax[4].set(title="Greedy evaluation", xlabel="Eval episode", ylabel="Return", ylim=(0, 520))
plt.tight_layout(); plt.show()

# ---------- 12.6 RESULTS ----------
print(f"Episodes completed: {len(rets_)} | best 30-episode average: {smooth(rets_, 30).max():.1f}")
print(f"Final greedy evaluation: {np.mean(eval_returns):.1f} ± {np.std(eval_returns):.1f}")

# ---------- 12.7 FINAL SUMMARY ----------
import textwrap
print("\n" + "=" * 100 + "\nSUMMARY\n" + "=" * 100)
print(textwrap.fill(
    f"PPO collected {N_STEPS} steps from each of {N_ENVS} environments, computed GAE(λ={GAE_LAMBDA}) advantages and reused each batch for {EPOCHS} epochs of clipped-objective mini-batch updates (clip ε={CLIP}), "
    f"for {venv.t} environment steps and {len(rets_)} finished episodes. The best 30-episode average return was {smooth(rets_, 30).max():.1f} and the greedy evaluation over 10 episodes averaged "
    f"{np.mean(eval_returns):.1f} ± {np.std(eval_returns):.1f} (max 500). The diagnostics show the trust-region idea at work: the average clip fraction was {np.mean(logs['clipfrac']):.2f} and the approximate KL stayed small "
    f"(mean {np.mean(logs['kl']):.4f}), so each update changed the policy only a little, which is why PPO's learning curve is smoother and more reliable than REINFORCE, actor-critic or A2C.", 100))

**Summary of Experiment 12.** WHAT WE DID: We implemented Proximal Policy Optimization. Rollouts from 8 environments are turned into GAE
advantages, then reused for several epochs of mini-batch updates using the clipped surrogate objective min(ρA,
clip(ρ,1-ε,1+ε)A), which stops the new policy from straying too far from the policy that collected the data. A
linearly decaying learning rate is also used. HOW TO READ THE PLOTS: training return, clipped policy loss,
value loss, clip fraction and approximate KL (trust-region diagnostics), and a greedy evaluation. RESULT: PPO
is typically the smoothest and most reliable of the policy-gradient methods in this notebook, reaching or
approaching 500 on CartPole; the printed summary has this run's numbers.

## 🧾 Wrap-up: how the 12 experiments relate

| Method | Learns | Needs model? | Update from | Key idea | Weakness |
|---|---|---|---|---|---|
| Epsilon-greedy | action values (no state) | No | each pull | explore with prob. ε | no state/context |
| DP evaluation / Policy iteration | V, π | **Yes** | full sweeps | Bellman equations | needs model, small state spaces |
| Monte Carlo | V | No | full episodes | average returns | high variance, episodic only |
| TD(0) / Q-learning / SARSA | V / Q | No | every step | bootstrapping | tabular → small problems |
| DQN | Q (neural net) | No | replay batches | replay + target net | discrete actions only |
| TD-Gammon | V (neural net) | No | every move | TD(λ) + self-play | needs many games |
| REINFORCE | π | No | full episodes | policy gradient | high variance |
| Actor-Critic / A2C | π and V | No | every step / n steps | critic as baseline | sensitive to hyper-parameters |
| PPO | π and V | No | batches, reused | clipped ratio | more machinery, but the most robust |

**Ideas to go further:** change seeds and see the variance, try Q-learning on `FrozenLake`, swap CartPole for `LunarLander-v3`, or add a
baseline-vs-no-baseline comparison in REINFORCE.